# SCAG: cross-stage associations and intervention evidence
Run top to bottom in the **cinema** kernel. This focused notebook constructs the four-stage graph, retrieves concept-associated neighbourhoods, and tests one adjacent-stage dependency experiment. The previous notebook is preserved in a timestamped backup.

Training patients determine scores, graph edges, source/target selections and normalisation. Validation patients measure intervention effects; these already-inspected patients provide exploratory evidence. Test data remain off by default. Top labels are forced assignments, not verified detectors. Cross-stage correlation edges remain undirected; intervention results are exported separately.

Architecture selection is in the top configuration cell. DenseNet-121 and DenseNet-161 each have four dense blocks, captured at `encoder.features.denseblock1` through `denseblock4` (before transitions and final norm/ReLU). These are stage-output channels, not one-to-one counterparts of ResNet channels. Dense connectivity concatenates retained and newly generated features within a block. Switching models requires restarting the kernel and running all cells; outputs are separated by model and timestamp.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
from scipy.stats import rankdata
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scag" / "models.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the SCAG repository.")
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scag.config import SCAGConfig
from scag.data import load_datasets
from scag.models import build_model

# Edit this section, restart the kernel, then Run All.
MODEL = "densenet161"  # "resnet18", "densenet121", or "densenet161"; requires a trained checkpoint.
DATASET = "acdc"
MODEL_SEED = 0
DEVICE = "cuda"
BATCH_SIZE = 16  # Reduce for DenseNet if GPU memory is insufficient.
ALPHA = 0.005
GAMMA = 0.015  # Comparison threshold only; top-concept assignments remain forced.
TAU = 0.30
SEED = 42
EVALUATE_TEST = False
QUERY_CONCEPT = "LV"
CONCEPTS = ["RV", "MYO", "LV"]
ANALYSIS_QUERIES = [QUERY_CONCEPT]  # Use CONCEPTS for all three queries.
ML_EVAL_SPLIT = "val"
PATIENT_INDEX = 0
ML_SEEDS = 10
ML_CROSS_NEIGHBORS = 5
ML_PATCH_MAX_PER_LAYER = 5
ML_PATCH_SIZE = 64
SOURCE_STAGE = 2  # One-based adjacent stage pair for the original intervention experiment.
TARGET_STAGE = 3
SOURCE_K = 10
TARGET_K = 10
RANDOM_REPEATS = 20
BOOTSTRAPS = 2000
TR_STAGE_BUDGET = 10
TR_RANDOM_REPEATS = 10
VR_NEIGHBORS_PER_PARENT = ML_CROSS_NEIGHBORS

MODEL_STAGES = {
    "resnet18": [f"encoder.layer{i}" for i in range(1, 5)],
    "densenet121": [f"encoder.features.denseblock{i}" for i in range(1, 5)],
    "densenet161": [f"encoder.features.denseblock{i}" for i in range(1, 5)],
}
if MODEL not in MODEL_STAGES:
    raise ValueError(f"Choose a configured model: {list(MODEL_STAGES)}")
ML_LAYERS = MODEL_STAGES[MODEL].copy()
LAYER = ML_LAYERS[-1]
ML_QUERY = QUERY_CONCEPT
TR_QUERIES = list(ANALYSIS_QUERIES)
TR_PATIENT_INDEX = PATIENT_INDEX
VR_QUERIES = list(ANALYSIS_QUERIES)
VR_SEED_COUNT = TR_STAGE_BUDGET
VR_RANDOM_REPEATS = TR_RANDOM_REPEATS
VR_PATIENT_INDEX = PATIENT_INDEX
assert 1 <= SOURCE_STAGE < TARGET_STAGE <= len(ML_LAYERS) and TARGET_STAGE == SOURCE_STAGE + 1
SOURCE_LAYER, TARGET_LAYER = ML_LAYERS[SOURCE_STAGE-1], ML_LAYERS[TARGET_STAGE-1]
assert ML_EVAL_SPLIT in ("val", "test") and (ML_EVAL_SPLIT != "test" or EVALUATE_TEST)
assert QUERY_CONCEPT in CONCEPTS and ANALYSIS_QUERIES and all(c in CONCEPTS for c in ANALYSIS_QUERIES)
assert 0 < ALPHA < 1 and 0 <= GAMMA <= 1 and 0 < TAU <= 1
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this kernel. Use cinema with GPU access, or explicitly set DEVICE='cpu'.")
torch.set_num_threads(4)
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
RUN_DIR = ROOT / "scag_outputs" / "from_scratch" / f"{DATASET}_{MODEL}" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Device:", torch.cuda.get_device_name() if DEVICE == "cuda" else "CPU")
print("Outputs:", RUN_DIR)

def save_figure(fig, name):
    for extension in ["png", "pdf"]:
        fig.savefig(RUN_DIR / f"{name}.{extension}", dpi=180, bbox_inches="tight")
    plt.show()

## 1. Frozen model and patient-disjoint data

In [ ]:
cfg = SCAGConfig(dataset=DATASET, model_seed=MODEL_SEED)
checkpoint = cfg.model_root / MODEL / cfg.formulation / cfg.initialization / f"seed_{cfg.model_seed}" / "best_val_mcc.pt"
if not checkpoint.exists():
    raise FileNotFoundError(checkpoint)
checkpoint_sha256 = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
model, _, default_layer = build_model(MODEL, cfg)
model.to(DEVICE).eval()
assert cfg.formulation == "stacked", "The analysis requires a single joint ED/ES forward pass."
for stage_name in ML_LAYERS:
    model.get_submodule(stage_name)
print("Analysis stages:", ML_LAYERS)
print("Checkpoint default feature layer:", default_layer, "| selected final stage:", LAYER)
(RUN_DIR / "experiment_config.json").write_text(json.dumps(dict(model=MODEL, dataset=DATASET, model_seed=MODEL_SEED, checkpoint_sha256=checkpoint_sha256, layers=ML_LAYERS, queries=ANALYSIS_QUERIES, eval_split=ML_EVAL_SPLIT, alpha=ALPHA, gamma=GAMMA, tau=TAU, seed=SEED, source_stage=SOURCE_STAGE, target_stage=TARGET_STAGE, source_k=SOURCE_K, target_k=TARGET_K, random_repeats=RANDOM_REPEATS, bootstraps=BOOTSTRAPS, fixed_budget=TR_STAGE_BUDGET, traversal_random_repeats=TR_RANDOM_REPEATS, variable_neighbors=VR_NEIGHBORS_PER_PARENT, patient_index=PATIENT_INDEX), indent=2))
target_layer = model.get_submodule(LAYER)
loaded = load_datasets(cfg)
datasets = dict(zip(["train", "val", "test"], loaded[:3]))
all_manifests = {}
for split, dataset in datasets.items():
    frame = dataset.metadata[["pid", "pathology"]].astype(str).reset_index(drop=True)
    assert frame.pid.is_unique and len(frame) == len(dataset)
    frame.insert(0, "sample_index", np.arange(len(frame)))
    frame["split"] = split
    all_manifests[split] = frame
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not set(all_manifests[left].pid) & set(all_manifests[right].pid)
SPLITS = ["train", "val", "test"] if EVALUATE_TEST else ["train", "val"]
display(pd.DataFrame([dict(split=s, patients=len(d), analysed=s in SPLITS) for s, d in datasets.items()]))
print("Checkpoint:", checkpoint)
print("Layer:", LAYER)

In [ ]:
activations, images, segments, masks, manifests = {}, {}, {}, {}, {}
CONCEPTS = ["RV", "MYO", "LV"]
for split in SPLITS:
    captured, image_batches, segment_batches, observed_ids, observed_slices = [], [], [], [], []
    def capture(module, inputs, output):
        if not torch.is_tensor(output) or output.ndim != 4:
            raise ValueError("Selected layer must return N x channels x height x width.")
        captured.append(output.detach().to("cpu", copy=True))
    handle = target_layer.register_forward_hook(capture)
    try:
        with torch.inference_mode():
            loader = torch.utils.data.DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            for x, segmentation, metadata in loader:
                before = len(captured)
                model(x.to(DEVICE))
                assert len(captured) == before + 1, "Layer must be called exactly once per forward pass."
                image_batches.append(x.cpu())
                segment_batches.append(segmentation.cpu().to(torch.int16))
                observed_ids.extend(str(pid) for pid in metadata["pid"])
                if "slice_index" in metadata:
                    observed_slices.extend(int(v) for v in metadata["slice_index"])
    finally:
        handle.remove()
    assert observed_ids == all_manifests[split].pid.tolist(), "Loader and manifest order differ."
    activations[split] = torch.cat(captured)
    images[split] = torch.cat(image_batches)
    segments[split] = torch.cat(segment_batches)
    assert segments[split].ndim == 4 and segments[split].shape[1] == 2
    assert torch.all((segments[split] >= 0) & (segments[split] <= 3))
    assert torch.isfinite(activations[split]).all()
    masks[split] = torch.stack([(segments[split] == value).any(dim=1) for value in [1, 2, 3]], dim=1)
    assert masks[split].shape[-2:] == images[split].shape[-2:]
    manifests[split] = all_manifests[split].copy()
    if observed_slices:
        assert len(observed_slices) == len(manifests[split])
        manifests[split]["slice_index"] = observed_slices
    manifests[split].to_csv(RUN_DIR / f"{split}_samples.csv", index=False)
    np.savez_compressed(RUN_DIR / f"spatial_evidence_{split}.npz", activations=activations[split].numpy(), images=images[split].numpy(), masks=masks[split].numpy(), pid=np.asarray(observed_ids))
    print(f"{split}: activations {tuple(activations[split].shape)}, masks {tuple(masks[split].shape)}")
CHANNELS = activations["train"].shape[1]
assert all(a.shape[1] == CHANNELS for a in activations.values())

## 2. Shared association and coactivation definitions

In [ ]:
def association_profiles(acts, concept_masks, thresholds):
    n, channels = acts.shape[:2]
    assert concept_masks.ndim == 4 and len(concept_masks) == n
    assert thresholds.shape == (channels,) and np.isfinite(thresholds).all()
    intersection = np.zeros((n, channels, concept_masks.shape[1]), dtype=np.float64)
    union = np.zeros_like(intersection)
    active_area = np.zeros((n, channels), dtype=np.float64)
    cutoff = torch.as_tensor(thresholds, dtype=acts.dtype, device=DEVICE)
    with torch.inference_mode():
        for start in range(0, n, 8):
            end = min(start + 8, n)
            targets = concept_masks[start:end].to(DEVICE).float().flatten(2)
            target_area = targets.sum(dim=2)
            for channel in range(0, channels, 16):
                stop = min(channel + 16, channels)
                up = torch.nn.functional.interpolate(acts[start:end, channel:stop].to(DEVICE), size=concept_masks.shape[-2:], mode="bilinear", align_corners=False)
                active = (up > cutoff[channel:stop][None, :, None, None]).float().flatten(2)
                overlap = torch.bmm(active, targets.transpose(1, 2))
                area = active.sum(dim=2)[:, :, None] + target_area[:, None, :] - overlap
                intersection[start:end, channel:stop] = overlap.cpu().numpy()
                union[start:end, channel:stop] = area.cpu().numpy()
                active_area[start:end, channel:stop] = active.sum(dim=2).cpu().numpy()
    total_union = union.sum(axis=0)
    scores = np.divide(intersection.sum(axis=0), total_union, out=np.zeros_like(total_union), where=total_union > 0)
    patient_iou = np.divide(intersection, union, out=np.zeros_like(union), where=union > 0)
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    return dict(S=scores, per_patient_iou=patient_iou, intersection=intersection, union=union, active_area=active_area)

def coactivation_matrix(acts):
    pooled = acts.mean(dim=(2, 3)).numpy()
    assert len(pooled) >= 2 and np.isfinite(pooled).all()
    ranks = rankdata(pooled, axis=0)
    ranks -= ranks.mean(axis=0)
    norms = np.linalg.norm(ranks, axis=0)
    normalized = np.divide(ranks, norms, out=np.zeros_like(ranks), where=norms > 0)
    correlation = np.clip(normalized.T @ normalized, -1, 1)
    np.fill_diagonal(correlation, 0)
    assert np.allclose(correlation, correlation.T) and np.isfinite(correlation).all()
    return correlation, pooled

def label_neurons(scores, names, gamma):
    scores = np.asarray(scores)
    if scores.ndim != 2 or scores.shape[1] != len(names) or len(names) < 2 or len(set(names)) != len(names):
        raise ValueError("Require one score column per unique concept.")
    if not np.isfinite(scores).all() or (scores < 0).any() or (scores > 1).any() or not 0 <= gamma <= 1:
        raise ValueError("Scores and threshold must be finite and in [0,1].")
    best = scores.argmax(axis=1)
    top = scores[np.arange(len(scores)), best]
    runner_up = np.sort(scores, axis=1)[:, -2]
    labels = np.asarray(names)[best]
    return pd.DataFrame(dict(neuron=np.arange(len(scores)), concept_index=best, label=labels, top_iou=top, runner_up_iou=runner_up, margin=top - runner_up, old_label=np.where(top >= gamma, labels, "unlabelled"), meets_old_threshold=top >= gamma, tie=(scores == top[:, None]).sum(axis=1) > 1, all_zero=(scores == 0).all(axis=1)))

def crop_window(peak_y, peak_x, height, width, size):
    crop_h, crop_w = min(size, height), min(size, width)
    y0 = min(max(int(peak_y) - crop_h // 2, 0), height - crop_h)
    x0 = min(max(int(peak_x) - crop_w // 2, 0), width - crop_w)
    return y0, y0 + crop_h, x0, x0 + crop_w

## 3. Extract all four stages

In [ ]:
assert ML_QUERY in CONCEPTS and ML_EVAL_SPLIT in SPLITS and ML_EVAL_SPLIT != "train"
ML_DIR = RUN_DIR / "multilayer_scag" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
ML_DIR.mkdir(parents=True)
ml_splits = ["train", ML_EVAL_SPLIT]
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == checkpoint_sha256
model.eval()
ml_acts = {split:{} for split in ml_splits}
for split in ml_splits:
    captured = {layer:[] for layer in ML_LAYERS}
    handles = []
    def make_stage_hook(layer):
        def hook(module, inputs, result):
            if not torch.is_tensor(result) or result.ndim != 4:
                raise ValueError(f"{layer} must return a spatial NCHW tensor.")
            captured[layer].append(result.detach().to("cpu", copy=True))
        return hook
    try:
        for layer in ML_LAYERS:
            handles.append(model.get_submodule(layer).register_forward_hook(make_stage_hook(layer)))
        with torch.inference_mode():
            for start in range(0,len(images[split]),BATCH_SIZE):
                before = {layer:len(captured[layer]) for layer in ML_LAYERS}
                model(images[split][start:start+BATCH_SIZE].to(DEVICE))
                assert all(len(captured[layer]) == before[layer]+1 for layer in ML_LAYERS)
    finally:
        for handle in handles:handle.remove()
    for layer in ML_LAYERS:
        ml_acts[split][layer] = torch.cat(captured[layer])
        assert len(ml_acts[split][layer]) == len(manifests[split])
        assert torch.isfinite(ml_acts[split][layer]).all()
    np.savez_compressed(ML_DIR / f"activations_{split}.npz", **{layer:a.numpy() for layer,a in ml_acts[split].items()}, pid=manifests[split].pid.to_numpy(dtype=str))
    manifests[split].to_csv(ML_DIR / f"{split}_samples.csv",index=False)
    print(split, {layer:tuple(a.shape) for layer,a in ml_acts[split].items()},flush=True)

## 4. Semantic profiles and within-/cross-stage coactivation

In [ ]:
def normalized_patient_ranks(values):
    values = np.asarray(values)
    if values.ndim != 2 or len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("Require finite patient x channel responses for at least two patients.")
    ranked = rankdata(values,axis=0)
    ranked -= ranked.mean(axis=0)
    norms = np.linalg.norm(ranked,axis=0)
    return np.divide(ranked,norms,out=np.zeros_like(ranked),where=norms>0)

# Check positive/negative correspondence and constant-channel behaviour.
check = np.array([[0.,2.],[1.,2.],[2.,2.],[3.,2.]])
np.testing.assert_allclose(normalized_patient_ranks(check).T @ normalized_patient_ranks(check),[[1,0],[0,0]],atol=1e-12)
assert np.isclose((normalized_patient_ranks(check[:,:1]).T @ normalized_patient_ranks(-check[:,:1])).item(),-1)
ml_profiles,ml_scores,ml_within,ml_ranks,ml_cutoffs,ml_labels = {},{},{},{},{},{}
ml_arrays = {"concepts":np.asarray(CONCEPTS)}
for layer in ML_LAYERS:
    train = ml_acts["train"][layer]
    count = train.shape[1]
    cutoff = np.percentile(train.numpy().reshape(len(train),count,-1),100*(1-ALPHA),axis=(0,2))
    ml_cutoffs[layer] = cutoff
    ml_arrays[f"thresholds_{layer}"] = cutoff
    for split in ml_splits:
        profile = association_profiles(ml_acts[split][layer],masks[split],cutoff)
        ml_profiles[(split,layer)] = profile
        ml_scores[(split,layer)] = profile["S"]
        correlation,responses = coactivation_matrix(ml_acts[split][layer])
        ml_within[(split,layer)] = correlation
        ml_ranks[(split,layer)] = normalized_patient_ranks(responses)
        ml_arrays.update({f"S_{split}_{layer}":profile["S"],f"R_{split}_{layer}":correlation,f"pooled_{split}_{layer}":responses})
    ml_labels[layer] = label_neurons(ml_scores[("train",layer)],CONCEPTS,GAMMA)
    assert ml_labels[layer].label.isin(CONCEPTS).all() and len(ml_labels[layer]) == count
    print(layer, "channels:",count,"assigned:",len(ml_labels[layer]),flush=True)
ml_cross = {}
for lower,upper_layer in zip(ML_LAYERS[:-1],ML_LAYERS[1:]):
    for split in ml_splits:
        cross = np.clip(ml_ranks[(split,lower)].T @ ml_ranks[(split,upper_layer)],-1,1)
        assert cross.shape == (len(ml_labels[lower]),len(ml_labels[upper_layer])) and np.isfinite(cross).all()
        ml_cross[(split,lower,upper_layer)] = cross
        ml_arrays[f"cross_{split}_{lower}__{upper_layer}"] = cross
np.savez_compressed(ML_DIR / "representations.npz", **ml_arrays)

In [ ]:
ml_graph = nx.Graph(description="Multilayer SCAG: forced concept assignments and measured undirected within/adjacent-layer coactivation")
ml_node_rows,ml_edge_rows = [],[]
for concept in CONCEPTS:
    ml_graph.add_node(f"concept:{concept}",kind="concept",label=concept)
for stage,layer in enumerate(ML_LAYERS):
    for row in ml_labels[layer].itertuples():
        node = f"{layer}:n{row.neuron}"
        attrs = dict(kind="neuron",layer=layer,stage=stage,neuron=int(row.neuron),label=row.label,top_iou=float(row.top_iou),tie=bool(row.tie),all_zero=bool(row.all_zero),meets_old_threshold=bool(row.meets_old_threshold))
        for c,name in enumerate(CONCEPTS):
            attrs[f"train_{name}"] = float(ml_scores[("train",layer)][row.neuron,c])
            attrs[f"{ML_EVAL_SPLIT}_{name}"] = float(ml_scores[(ML_EVAL_SPLIT,layer)][row.neuron,c])
        ml_graph.add_node(node,**attrs)
        ml_graph.add_edge(node,f"concept:{row.label}",kind="semantic_assignment",weight=float(row.top_iou))
        ml_node_rows.append(dict(node=node,**attrs))
    corr = ml_within[("train",layer)]
    for i,j in zip(*np.where(np.triu((corr>=TAU)&(corr>0),1))):
        a,b = f"{layer}:n{i}",f"{layer}:n{j}"
        attrs = dict(kind="within_layer_coactivation",weight=float(corr[i,j]),eval_r=float(ml_within[(ML_EVAL_SPLIT,layer)][i,j]))
        ml_graph.add_edge(a,b,**attrs)
        ml_edge_rows.append(dict(source=a,target=b,**attrs))
for lower,upper_layer in zip(ML_LAYERS[:-1],ML_LAYERS[1:]):
    corr = ml_cross[("train",lower,upper_layer)]
    for i,j in zip(*np.where((corr>=TAU)&(corr>0))):
        a,b = f"{lower}:n{i}",f"{upper_layer}:n{j}"
        attrs = dict(kind="cross_layer_coactivation",weight=float(corr[i,j]),eval_r=float(ml_cross[(ML_EVAL_SPLIT,lower,upper_layer)][i,j]))
        ml_graph.add_edge(a,b,**attrs)
        ml_edge_rows.append(dict(source=a,target=b,**attrs))
ml_nodes = pd.DataFrame(ml_node_rows)
ml_edges = pd.DataFrame(ml_edge_rows)
assert ml_nodes.node.is_unique and ml_nodes.label.isin(CONCEPTS).all()
assert sum(d["kind"]=="semantic_assignment" for _,_,d in ml_graph.edges(data=True)) == len(ml_nodes)
ml_nodes.to_csv(ML_DIR / "neuron_labels.csv",index=False)
ml_edges.to_csv(ML_DIR / "functional_edges.csv.gz",index=False)
nx.write_graphml(ml_graph,ML_DIR / "joint_graph.graphml.gz")
display(ml_nodes.groupby(["layer","label"]).size().unstack(fill_value=0))
display(ml_edges.groupby("kind").agg(training_edges=("weight","size"),evaluation_retained=("eval_r",lambda r:int((r>=TAU).sum()))))

## 5. Concept-guided neighbourhood
Retrieve up to `ML_SEEDS` final-stage seeds, then up to `ML_CROSS_NEIGHBORS` preceding-stage neighbours per channel. Selection uses training data only. Earlier-stage neighbours may carry different concept labels.

In [ ]:
last = ML_LAYERS[-1]
ml_seeds = ml_labels[last][ml_labels[last].label==ML_QUERY].sort_values(["top_iou","neuron"],ascending=[False,True]).head(ML_SEEDS).neuron.to_numpy(dtype=int).tolist()
if not ml_seeds:
    raise ValueError(f"No final-stage channels are assigned to {ML_QUERY}.")
ml_selected = {last:set(ml_seeds)}
ml_display_edges = set()
for index in range(len(ML_LAYERS)-2,-1,-1):
    lower,upper_layer = ML_LAYERS[index:index+2]
    corr = ml_cross[("train",lower,upper_layer)]
    chosen = set()
    for j in sorted(ml_selected[upper_layer]):
        candidates = np.flatnonzero((corr[:,j]>=TAU)&(corr[:,j]>0)).tolist()
        neighbors = sorted(candidates,key=lambda i:(-corr[i,j],i))[:ML_CROSS_NEIGHBORS]
        for i in neighbors:
            chosen.add(i)
            ml_display_edges.add(tuple(sorted((f"{lower}:n{i}",f"{upper_layer}:n{j}"))))
    ml_selected[lower] = chosen
for layer in ML_LAYERS:
    for i in sorted(ml_selected[layer]):
        neighbors = [j for j in ml_selected[layer] if j!=i and ml_graph.has_edge(f"{layer}:n{i}",f"{layer}:n{j}")]
        if neighbors:
            j = sorted(neighbors,key=lambda j:(-ml_within[("train",layer)][i,j],j))[0]
            ml_display_edges.add(tuple(sorted((f"{layer}:n{i}",f"{layer}:n{j}"))))
ml_selected_nodes = {f"{layer}:n{i}" for layer in ML_LAYERS for i in ml_selected[layer]}
assert all(ml_graph.has_edge(a,b) for a,b in ml_display_edges)
ml_display_table = pd.DataFrame([dict(source=a,target=b,**ml_graph[a][b]) for a,b in sorted(ml_display_edges)])
ml_display_table.to_csv(ML_DIR / "displayed_edges.csv",index=False)
ml_nodes[ml_nodes.node.isin(ml_selected_nodes)].to_csv(ML_DIR / "displayed_nodes.csv",index=False)


In [ ]:
"""Display the frozen training-selected neighbourhood."""
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Rectangle
import numpy as np

# --------- Publication style ---------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
})
ml_palette = {"RV": "#2d6cab", "MYO": "#3bab3b", "LV": "#dd6d6d"}
ink = "#273547"
plane_color = "#EEF2F6"

# Context channels are displayed only to indicate layer scale and the
# distribution of annotation labels. No displayed neighbourhood node/edge is sampled.
# Set to None to draw every background channel (less legible in print).
BACKGROUND_MAX = 80
stage_gap = 1.45

# Use the graph-construction threshold without changing it during plotting.

# Positions are purely illustrative. Barycentric ordering reduces edge
# crossings, but it must NOT be interpreted as anatomical position or similarity.
ml_positions = {}
context_by_layer = {}
for stage, layer in enumerate(ML_LAYERS):
    layer_ids = sorted(ml_nodes.loc[ml_nodes.layer == layer, "node"].astype(str).tolist())
    total = len(layer_ids)
    # Vogel spiral for the faint contextual channel cloud.
    for k, node in enumerate(layer_ids):
        rho = np.sqrt((k + .5) / max(total, 1))
        phi = k * np.pi * (3 - np.sqrt(5))
        ml_positions[node] = (0.98 * rho * np.cos(phi),
                              0.72 * rho * np.sin(phi), stage_gap * stage)
    others = [n for n in layer_ids if n not in ml_selected_nodes]
    if BACKGROUND_MAX is not None and len(others) > BACKGROUND_MAX:
        rng = np.random.default_rng(SEED + 7919 * (stage + 1))
        keep = np.sort(rng.choice(len(others), size=BACKGROUND_MAX, replace=False))
        others = [others[k] for k in keep]
    context_by_layer[layer] = others

# Ordering selected channels top-down using displayed CROSS-layer edges.
# This changes only the visualization coordinates, not the measured graph.
orders = {}
last = ML_LAYERS[-1]
orders[last] = [f"{last}:n{i}" for i in ml_seeds]
selected_edge_set = set(ml_display_edges)
for stage in range(len(ML_LAYERS) - 2, -1, -1):
    lower, upper = ML_LAYERS[stage:stage + 2]
    upper_rank = {n: j for j, n in enumerate(orders[upper])}
    current = [f"{lower}:n{i}" for i in sorted(ml_selected[lower])]

    def neighbour_barycentre(node):
        matching = [upper_rank[u] for u in orders[upper]
                    if tuple(sorted((node, u))) in selected_edge_set]
        return np.mean(matching) if matching else float("inf")

    orders[lower] = sorted(current, key=lambda n: (neighbour_barycentre(n), n))

for stage, layer in enumerate(ML_LAYERS):
    ordered = orders[layer]
    if not ordered:
        continue
    spread = np.linspace(-.82, .82, len(ordered)) if len(ordered) > 1 else [0.]
    for j, (node, x) in enumerate(zip(ordered, spread)):
        # Small third-dimensional jitter separates overlaps in the oblique view.
        y = .21 * np.sin(j * 2.39996 + stage)
        ml_positions[node] = (float(x), float(y), stage_gap * stage)

seed_ids = {f"{last}:n{i}" for i in ml_seeds}
node_label = ml_nodes.drop_duplicates("node").set_index("node")["label"].to_dict()

# Only retained training-selected display edges are drawn; evaluation status
# controls stroke pattern, not whether an edge is included.
def edge_style(a, b):
    e = ml_graph[a][b]
    eval_r = float(e["eval_r"])
    surviving = bool(np.isfinite(eval_r) and eval_r >= TAU)
    same_stage = a.split(":n")[0] == b.split(":n")[0]
    train_r = float(e["weight"])
    return {
        "color": ("#8795A5" if same_stage else ink) if surviving else "#B3BBC6",
        "linestyle": "-" if surviving else (0, (3.0, 2.0)),
        "linewidth": .56 + .85 * np.clip(train_r, 0, 1),
        "alpha": (.75 if surviving else .8),
    }

fig = plt.figure(figsize=(7.4, 4.45), facecolor="white")
ax2 = fig.add_axes([.03, .20, .48, .72])
ax3 = fig.add_axes([.50, .17, .47, .75], projection="3d")

# ---------------- Panel A: readable stage hierarchy ----------------
for stage, layer in enumerate(ML_LAYERS):
    z = stage_gap * stage
    ax2.add_patch(Rectangle((-1.02, z - .19), 2.04, .38,
                            facecolor=plane_color, edgecolor="#DDE3EB", lw=.5,
                            zorder=0))
    faint = context_by_layer[layer]
    if faint:
        pts = np.array([ml_positions[n] for n in faint])
        colors = [ml_palette.get(node_label.get(n), "#9DA7B3") for n in faint]
        ax2.scatter(pts[:, 0], pts[:, 2] + .18 * pts[:, 1],
                    c=colors, s=3.5, alpha=.22, linewidths=0, zorder=1,
                    rasterized=True)
    ax2.text(-1.12, z, f"Layer {stage + 1}",
             ha="right", va="center", size=7.1, color=ink)

# Edges under nodes; curved arcs identify within-stage connections.
for a, b in sorted(ml_display_edges):
    pa, pb = ml_positions[a], ml_positions[b]
    xy1 = (pa[0], pa[2] + .18 * pa[1])
    xy2 = (pb[0], pb[2] + .18 * pb[1])
    sty = edge_style(a, b)
    if abs(pa[2] - pb[2]) < 1e-9:
        artist = FancyArrowPatch(xy1, xy2, arrowstyle="-",
                                 connectionstyle="arc3,rad=.33",
                                 color=sty["color"], lw=sty["linewidth"],
                                 linestyle=sty["linestyle"], alpha=sty["alpha"],
                                 mutation_scale=1, zorder=3)
        ax2.add_patch(artist)
    else:
        ax2.plot([xy1[0], xy2[0]], [xy1[1], xy2[1]],
                 zorder=2, solid_capstyle="round", **sty)

for stage, layer in enumerate(ML_LAYERS):
    for node in orders[layer]:
        x, y, z = ml_positions[node]
        is_seed = node in seed_ids
        ax2.scatter(x, z + .18 * y, s=53 if is_seed else 31,
                    c=ml_palette.get(node_label.get(node), "#9DA7B3"),
                    edgecolors="#1F2937" if is_seed else "white",
                    linewidths=1.0 if is_seed else .75, zorder=5)

ax2.set_xlim(-1.76, 1.11)
ax2.set_ylim(-.38, stage_gap * (len(ML_LAYERS) - 1) + .42)
ax2.set_aspect("auto")
ax2.axis("off")
ax2.set_title(f"(a) {ML_QUERY} neighbourhood", loc="left", color=ink, weight="bold", pad=7)

# ---------------- Panel B: orthographic layered perspective ----------------
xx, yy = np.meshgrid([-1.04, 1.04], [-.79, .79])
for stage, layer in enumerate(ML_LAYERS):
    z = stage_gap * stage
    ax3.plot_surface(xx, yy, np.full_like(xx, z),
                     color="#DDE5ED", alpha=.10, shade=False,
                     linewidth=0, antialiased=False)
    faint = context_by_layer[layer]
    if faint:
        xyz = np.array([ml_positions[n] for n in faint])
        colors = [ml_palette.get(node_label.get(n), "#9DA7B3") for n in faint]
        ax3.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2],
                    c=colors, s=4.2, alpha=.19, edgecolors="none",
                    depthshade=False, rasterized=True)

for a, b in sorted(ml_display_edges):
    p, q = ml_positions[a], ml_positions[b]
    sty = edge_style(a, b)
    ax3.plot([p[0], q[0]], [p[1], q[1]], [p[2], q[2]], **sty)

for layer in ML_LAYERS:
    ordinary = [n for n in orders[layer] if n not in seed_ids]
    special = [n for n in orders[layer] if n in seed_ids]
    for nodes, size, stroke in [(ordinary, 24, "white"), (special, 43, "#243044")]:
        if not nodes:
            continue
        xyz = np.array([ml_positions[n] for n in nodes])
        ax3.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2],
                    c=[ml_palette.get(node_label.get(n), "#9DA7B3") for n in nodes],
                    s=size, edgecolors=stroke, linewidths=.6,
                    depthshade=False)

ax3.view_init(elev=21, azim=-61)
ax3.set_proj_type("ortho")
ax3.set(xlim=(-1.06, 1.06), ylim=(-.88, .88),
        zlim=(-.12, stage_gap * (len(ML_LAYERS) - 1) + .12))
ax3.set_box_aspect((1.35, 1, 2.25), zoom=1.16)
ax3.set_axis_off()
ax3.set_title("(b) Layered view", loc="left", color=ink, weight="bold", pad=7)

# Compact shared legend; actual scientific caption goes in LaTeX, not inside art.
handles = [Line2D([0], [0], marker="o", linestyle="None", label=c,
                  markerfacecolor=ml_palette[c], markeredgecolor="white",
                  markersize=6) for c in ["RV", "MYO", "LV"]]
handles += [
    Line2D([0], [0], marker="o", linestyle="None", label=f"{ML_QUERY} seeds",
           markerfacecolor=ml_palette[ML_QUERY], markeredgecolor=ink, markersize=7),
    # Line2D([0], [0], color=ink, lw=1.3, linestyle="-", label="Retained on eval"),
    # Line2D([0], [0], color="#9EA8B5", lw=1.2, linestyle="--",
    #        label="Below eval threshold"),
]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, .105),
           ncol=6, frameon=False, handlelength=1.8, handletextpad=.45,
           columnspacing=1.1, fontsize=7)


ML_DIR.mkdir(parents=True, exist_ok=True)
for ext in ("pdf", "png", "svg"):
    fig.savefig(ML_DIR / f"{ML_QUERY}_layered_paper.{ext}",
                dpi=400, bbox_inches="tight", pad_inches=.035)
plt.show()

print("Highlighted channels:", {l: len(ml_selected[l]) for l in ML_LAYERS})
print("Displayed edges retained on evaluation:",
      int((ml_display_table.eval_r >= TAU).sum()), "/", len(ml_display_table))


## 6. Visual evidence on graph nodes
Training examples illustrate the selected channels. These examples can come from different patients and are not a single prediction pathway.

In [ ]:
from matplotlib.offsetbox import AnnotationBbox, OffsetImage, TextArea, VPacker
from matplotlib.colors import Normalize
ml_detail = {last:{ml_seeds[0]}}
for index in range(len(ML_LAYERS)-2,-1,-1):
    lower,upper_layer = ML_LAYERS[index:index+2]
    corr = ml_cross[("train",lower,upper_layer)]
    parents = sorted(ml_detail[upper_layer])
    candidates = set()
    for j in parents:
        eligible = np.flatnonzero((corr[:,j]>=TAU)&(corr[:,j]>0)).tolist()
        candidates.update(sorted(eligible,key=lambda i:(-corr[i,j],i))[:ML_CROSS_NEIGHBORS])
    ml_detail[lower] = set(sorted(candidates,key=lambda i:(-float(corr[i,parents].max()),i))[:ML_PATCH_MAX_PER_LAYER])
ml_detail_nodes = [f"{layer}:n{i}" for layer in ML_LAYERS for i in sorted(ml_detail[layer])]
ml_detail_graph = ml_graph.subgraph(ml_detail_nodes).copy()
assert all(node in ml_selected_nodes for node in ml_detail_nodes)
nx.write_graphml(ml_detail_graph,ML_DIR / "patch_detail.graphml")
ml_patch_rows,ml_patch_arrays,ml_patch_nodes = [],[],[]
ml_detail_positions = {}
fig,ax = plt.subplots(figsize=(14,11))
ax.set(xlim=(0,1),ylim=(-.04,1.08))
ax.axis("off")
for stage,layer in enumerate(ML_LAYERS):
    node_ids = [f"{layer}:n{i}" for i in sorted(ml_detail[layer])]
    xs = np.linspace(.22,.83,len(node_ids)) if len(node_ids)>1 else [.525]
    y = .12+.245*stage
    ax.axhline(y,color="#eeeeee",linewidth=1,zorder=0)
    ax.text(.01,y,f"layer{stage+1}",va="center",fontsize=11,color="#666666")
    for node,x in zip(node_ids,xs):ml_detail_positions[node]=(float(x),y)
for a,b,data in ml_detail_graph.edges(data=True):
    x=[ml_detail_positions[a][0],ml_detail_positions[b][0]]
    y=[ml_detail_positions[a][1],ml_detail_positions[b][1]]
    ax.plot(x,y,color="#888888",linewidth=.7+1.2*data["weight"],alpha=.75,linestyle="-" if data["eval_r"]>=TAU else "--",zorder=1)
for node in ml_detail_nodes:
    attrs = ml_graph.nodes[node]
    layer,neuron = attrs["layer"],attrs["neuron"]
    native = ml_acts["train"][layer][:,neuron].numpy()
    valid = np.flatnonzero(np.ptp(native,axis=(1,2))>0)
    title = TextArea(f"L{attrs['stage']+1} · n{neuron} · {attrs['label']}\ntraining IoU {attrs['top_iou']:.4f}",textprops=dict(ha="center",fontsize=9))
    if len(valid):
        sample = sorted(valid.tolist(),key=lambda i:(-float(native[i].max()),i))[0]
        height,width = images["train"].shape[-2:]
        response = torch.nn.functional.interpolate(ml_acts["train"][layer][sample:sample+1,neuron:neuron+1],size=(height,width),mode="bilinear",align_corners=False)[0,0].numpy()
        py,px = np.unravel_index(response.argmax(),response.shape)
        y0,y1,x0,x1 = crop_window(py,px,height,width,ML_PATCH_SIZE)
        pair = images["train"][sample,:,y0:y1,x0:x1].numpy().copy()
        canvas = np.concatenate([pair[0],pair[1]],axis=1)
        image_box = OffsetImage(canvas,cmap="gray",norm=Normalize(vmin=float(images['train'][sample].min()),vmax=max(float(images['train'][sample].max()),1e-8)),zoom=.95)
        pid = str(manifests["train"].iloc[sample].pid)
        caption = TextArea(f"{pid}   ED | ES",textprops=dict(ha="center",fontsize=8))
        box = VPacker(children=[title,image_box,caption],align="center",pad=2,sep=3)
        ml_patch_rows.append(dict(node=node,layer=layer,neuron=neuron,assigned_concept=attrs["label"],sample_index=sample,pid=pid,x0=x0,x1=x1,y0=y0,y1=y1,native_peak=float(native[sample].max()),upsampled_peak=float(response.max()),peak_tie_pixels=int((response==response.max()).sum()),training_iou=attrs["top_iou"],patient_iou=float(ml_profiles[("train",layer)]["per_patient_iou"][sample,neuron,CONCEPTS.index(attrs["label"])])))
        ml_patch_arrays.append(pair)
        ml_patch_nodes.append(node)
        np.testing.assert_array_equal(pair,images["train"][sample,:,y0:y1,x0:x1].numpy())
    else:
        box = VPacker(children=[title,TextArea("No spatially varying response",textprops=dict(fontsize=8))],align="center",pad=3,sep=4)
        ml_patch_rows.append(dict(node=node,layer=layer,neuron=neuron,assigned_concept=attrs["label"],status="no spatial peak"))
    ax.add_artist(AnnotationBbox(box,ml_detail_positions[node],frameon=True,bboxprops=dict(edgecolor=ml_palette[attrs["label"]],facecolor="white",linewidth=2,boxstyle="round,pad=.25"),zorder=3))
ax.set_title(f"{ML_QUERY} query: cross-layer SCAG with MRI patch nodes\nNode borders: assigned concepts; edges: measured undirected coactivation",fontsize=14,pad=12)
fig.text(.5,.02,f"One activation-selected training example per channel; fixed {ML_PATCH_SIZE}×{ML_PATCH_SIZE} crops, not receptive fields.\nSolid edges retain the threshold on {ML_EVAL_SPLIT}; dashed edges do not. No causal flow is asserted.",ha="center",fontsize=10)
fig.subplots_adjust(top=.91,bottom=.06,left=.02,right=.98)
for extension in ["png","pdf"]:
    fig.savefig(ML_DIR / f"{ML_QUERY}_patch_nodes.{extension}",dpi=220,bbox_inches="tight")
plt.show()
pd.DataFrame(ml_patch_rows).to_csv(ML_DIR / "patch_evidence.csv",index=False)
if ml_patch_arrays:
    np.savez_compressed(ML_DIR / "patch_arrays.npz",images=np.stack(ml_patch_arrays),nodes=np.asarray(ml_patch_nodes))

## 7. Freeze the intervention experiment
Default: **layer2 → layer3**, one queried concept, ten source channels per method and ten downstream targets. SCAG ranks source channels by their query IoU multiplied by their weighted coactivation with query-associated downstream channels. This is a graph-guided group selection test, distinct from the illustrative backward traversal above.

Semantic-only ranks query IoU; functional-only ranks within-stage weighted degree; random selects equal-sized groups. All use the same source layer, inputs and intervention. The primary baseline is channel zeroing; training-mean replacement is a sensitivity check.

For each source set, graph-connected targets are ranked by total retained cross-stage weight. Equal-sized random samples of targets having **no retained edge to any selected source** are controls. If insufficient such targets exist, the comparison is reported unavailable, without relaxing the threshold. Target effects are normalised by each channel's training pooled-response standard deviation; constant channels are excluded. Controls are not degree/activation matched, so a difference is not proof of path specificity.


### Freeze selections


In [ ]:
assert ML_LAYERS.index(TARGET_LAYER) == ML_LAYERS.index(SOURCE_LAYER) + 1
assert RANDOM_REPEATS > 0 and BOOTSTRAPS > 0
INTERVENTION_DIR = ML_DIR / "interventions"
INTERVENTION_DIR.mkdir(exist_ok=True)
source_train = ml_acts["train"][SOURCE_LAYER].mean((2, 3)).numpy()
target_train = ml_acts["train"][TARGET_LAYER].mean((2, 3)).numpy()
target_sd = target_train.std(axis=0, ddof=1)
valid_targets = np.flatnonzero(target_sd > 1e-8)
concept_index = CONCEPTS.index(ML_QUERY)
source_semantic = ml_scores[("train", SOURCE_LAYER)][:, concept_index]
target_semantic = ml_scores[("train", TARGET_LAYER)][:, concept_index]
cross = ml_cross[("train", SOURCE_LAYER, TARGET_LAYER)]
cross_adj = np.where((cross >= TAU) & (cross > 0), cross, 0)
within = ml_within[("train", SOURCE_LAYER)]
functional_score = np.where((within >= TAU) & (within > 0), within, 0).sum(axis=1)
joint_score = source_semantic * (cross_adj @ target_semantic)
assert 0 < SOURCE_K <= len(source_semantic) and 0 < TARGET_K <= len(valid_targets)
if np.count_nonzero(joint_score > 0) < SOURCE_K:
    raise ValueError("Insufficient positive joint scores for SOURCE_K; prespecify a smaller budget and rerun.")
def top_channels(scores, count):
    return np.lexsort((np.arange(len(scores)), -np.asarray(scores)))[:count].astype(int)
assert top_channels(np.array([1., 1., 2.]), 2).tolist() == [2, 0]
rng = np.random.default_rng(SEED)
source_sets = {"scag": top_channels(joint_score, SOURCE_K), "semantic": top_channels(source_semantic, SOURCE_K), "functional": top_channels(functional_score, SOURCE_K)}
source_sets.update({f"random_{r}": np.sort(rng.choice(len(source_semantic), SOURCE_K, replace=False)) for r in range(RANDOM_REPEATS)})
target_sets, selection_rows = {}, []
for strategy, channels in source_sets.items():
    strength = cross_adj[channels].sum(axis=0)
    connected = valid_targets[strength[valid_targets] > 0]
    targets = connected[np.lexsort((connected, -strength[connected]))[:TARGET_K]]
    unconnected = valid_targets[strength[valid_targets] == 0]
    controls = [np.sort(rng.choice(unconnected, len(targets), replace=False)) for _ in range(RANDOM_REPEATS)] if len(targets) and len(unconnected) >= len(targets) else []
    target_sets[strategy] = (targets, controls)
    assert len(channels) == SOURCE_K and len(np.unique(channels)) == SOURCE_K
    assert all(not set(targets) & set(control) for control in controls)
    selection_rows.append(dict(strategy=strategy, sources=channels.tolist(), targets=targets.tolist(), unconnected_controls=[c.tolist() for c in controls], target_count=len(targets), control_available=bool(controls)))
selection_protocol = dict(query=ML_QUERY, source_layer=SOURCE_LAYER, target_layer=TARGET_LAYER, source_k=SOURCE_K, target_k=TARGET_K, tau=TAU, alpha=ALPHA, old_gamma=GAMMA, model=MODEL, dataset=DATASET, layers=ML_LAYERS, traversal_seeds=ML_SEEDS, traversal_neighbors=ML_CROSS_NEIGHBORS, label_policy="forced training argmax; fixed concept-order ties", split=ML_EVAL_SPLIT, seed=SEED, checkpoint_sha256=checkpoint_sha256, selections=selection_rows, normalisation="training SD of pooled target responses; SD<=1e-8 excluded", baselines=["zero", "train_mean"], inference="exploratory paired patient bootstrap; pointwise intervals; fixed model and selections")
(INTERVENTION_DIR / "selection_protocol.json").write_text(json.dumps(selection_protocol, indent=2))
display(pd.DataFrame(selection_rows)[["strategy", "target_count", "control_available"]])


### Intervention runner


In [ ]:

def run_stage_intervention(network, inputs, source_layer, target_layer, channels, replacement, batch_size, device, source_means=None):
    if source_layer == target_layer or inputs.ndim != 4 or len(inputs) == 0 or batch_size < 1:
        raise ValueError("Require distinct stages, nonempty NCHW inputs and positive batch size.")
    channels = np.asarray(channels, dtype=int)
    if channels.ndim != 1 or len(np.unique(channels)) != len(channels) or (channels < 0).any():
        raise ValueError("Channels must be unique nonnegative indices.")
    if replacement not in ("zero", "train_mean"):
        raise ValueError("Unknown intervention baseline.")
    captured, logits, handles = [], [], []
    def intervene(module, args, output):
        if output.ndim != 4 or (len(channels) and channels.max() >= output.shape[1]):
            raise ValueError("Invalid intervention channel or tensor shape.")
        changed = output.clone()
        if len(channels):
            idx = channels.tolist()
            if replacement == "zero":
                changed[:, idx] = 0
            else:
                if source_means is None or np.asarray(source_means).shape != (output.shape[1],) or not np.isfinite(source_means).all():
                    raise ValueError("Training means must match all source channels and be finite.")
                means = torch.as_tensor(np.asarray(source_means)[channels], device=output.device, dtype=output.dtype)
                changed[:, idx] = means[None, :, None, None]
        return changed
    def capture(module, args, output):
        captured.append(output.detach().mean((2, 3)).cpu().clone())
    try:
        handles.append(network.get_submodule(source_layer).register_forward_hook(intervene))
        handles.append(network.get_submodule(target_layer).register_forward_hook(capture))
        with torch.inference_mode():
            for start in range(0, len(inputs), batch_size):
                before = len(captured)
                logits.append(network(inputs[start:start + batch_size].to(device)).detach().cpu())
                assert len(captured) == before + 1, "Target hook must fire exactly once per batch."
    finally:
        for handle in handles:
            handle.remove()
    outputs, responses = torch.cat(logits), torch.cat(captured).numpy()
    assert torch.isfinite(outputs).all() and np.isfinite(responses).all()
    return outputs, responses

# A tiny known network checks downstream effects, batching and hook cleanup.
toy = torch.nn.Sequential(torch.nn.Conv2d(1, 2, 1, bias=False), torch.nn.Conv2d(2, 1, 1, bias=False), torch.nn.Flatten()).eval()
with torch.no_grad():
    toy[0].weight.fill_(1)
    toy[1].weight.fill_(1)
toy_x = torch.ones(3, 1, 2, 2)
toy_clean, toy_response = run_stage_intervention(toy, toy_x, "0", "1", [], "zero", 2, "cpu")
toy_changed, toy_after = run_stage_intervention(toy, toy_x, "0", "1", [0], "zero", 2, "cpu")
np.testing.assert_allclose(toy_response - toy_after, 1)
assert torch.equal(toy(toy_x), toy_clean) and not toy[0]._forward_hooks and not toy[1]._forward_hooks
try:
    run_stage_intervention(toy, toy_x, "0", "1", [5], "zero", 2, "cpu")
except ValueError:
    pass
else:
    raise AssertionError("Invalid channel must fail.")
assert not toy[0]._forward_hooks and not toy[1]._forward_hooks
print("Intervention self-checks passed.")


### Measure effects


In [ ]:
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == checkpoint_sha256
model.eval()
eval_inputs = images[ML_EVAL_SPLIT]
eval_ids = manifests[ML_EVAL_SPLIT].pid.astype(str).tolist()
assert len(eval_ids) == len(set(eval_ids)) == len(eval_inputs)
clean_logits, clean_target = run_stage_intervention(model, eval_inputs, SOURCE_LAYER, TARGET_LAYER, [], "zero", BATCH_SIZE, DEVICE)
np.testing.assert_allclose(clean_target, ml_acts[ML_EVAL_SPLIT][TARGET_LAYER].mean((2, 3)).numpy(), rtol=1e-4, atol=1e-5)
original_class = clean_logits.argmax(1)
class_names = list(cfg.classes)
original_logp = clean_logits.log_softmax(1)
original_probability = original_logp.exp()
rows, target_rows, intervention_cache = [], [], {}
for baseline in ["zero", "train_mean"]:
    for strategy, channels in source_sets.items():
        key = (baseline, tuple(sorted(channels.tolist())))
        if key not in intervention_cache:
            intervention_cache[key] = run_stage_intervention(model, eval_inputs, SOURCE_LAYER, TARGET_LAYER, channels, baseline, BATCH_SIZE, DEVICE, source_means=source_train.mean(axis=0))
        changed_logits, changed_target = intervention_cache[key]
        signed = np.divide(clean_target - changed_target, target_sd[None, :], out=np.full_like(clean_target, np.nan), where=target_sd[None, :] > 1e-8)
        targets, controls = target_sets[strategy]
        target_effect = np.abs(signed[:, targets]).mean(1) if len(targets) else np.full(len(eval_ids), np.nan)
        control_effect = np.mean([np.abs(signed[:, control]).mean(1) for control in controls], axis=0) if controls else np.full(len(eval_ids), np.nan)
        kl = (original_probability * (original_logp - changed_logits.log_softmax(1))).sum(1).clamp_min(0).numpy()
        index = torch.arange(len(eval_ids))
        logit_drop = (clean_logits[index, original_class] - changed_logits[index, original_class]).numpy()
        probability_after = changed_logits.softmax(1)[index, original_class].numpy()
        for patient, pid in enumerate(eval_ids):
            rows.append(dict(pid=pid, strategy=strategy, baseline=baseline, source_k=len(channels), target_k=len(targets), target_effect=float(target_effect[patient]), control_effect=float(control_effect[patient]), target_minus_control=float(target_effect[patient] - control_effect[patient]), kl=float(kl[patient]), logit_drop=float(logit_drop[patient]), probability_before=float(original_probability[patient, original_class[patient]]), probability_after=float(probability_after[patient]), original_class=class_names[int(original_class[patient])], flip=bool(changed_logits[patient].argmax() != original_class[patient])))
            for j in valid_targets:
                target_rows.append(dict(pid=pid, strategy=strategy, baseline=baseline, target=int(j), label=ml_labels[TARGET_LAYER].iloc[j].label, train_query_iou=float(target_semantic[j]), graph_connected=bool(cross_adj[channels, j].sum() > 0), selected_target=bool(j in targets), signed_change_sd=float(signed[patient, j])))
        print(baseline, strategy, flush=True)
    pd.DataFrame(rows).to_csv(INTERVENTION_DIR / "patient_effects.csv", index=False)
restored_logits, restored_target = run_stage_intervention(model, eval_inputs, SOURCE_LAYER, TARGET_LAYER, [], "zero", BATCH_SIZE, DEVICE)
torch.testing.assert_close(restored_logits, clean_logits)
np.testing.assert_allclose(restored_target, clean_target)
patient_effects = pd.DataFrame(rows)
pd.DataFrame(target_rows).to_csv(INTERVENTION_DIR / "target_effects.csv.gz", index=False)
np.savez_compressed(INTERVENTION_DIR / "clean_predictions.npz", logits=clean_logits.numpy(), pid=np.asarray(eval_ids))


### Patient bootstrap and comparison


In [ ]:
def patient_interval(values, repeats, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (float(values.mean()) if len(values) else np.nan, np.nan, np.nan, len(values))
    draw = np.random.default_rng(seed).integers(0, len(values), size=(repeats, len(values)))
    lower, upper = np.quantile(values[draw].mean(1), [.025, .975])
    return float(values.mean()), float(lower), float(upper), len(values)
assert patient_interval([2, 2, 2], 100, SEED) == (2., 2., 2., 3)
metrics = ["target_effect", "control_effect", "target_minus_control", "kl", "logit_drop", "flip"]
aggregated = patient_effects.assign(method=patient_effects.strategy.str.replace(r"random_\d+", "random", regex=True)).groupby(["pid", "baseline", "method"], as_index=False)[metrics].mean()
summary_rows = []
for (baseline, method), group in aggregated.groupby(["baseline", "method"]):
    for metric in metrics:
        mean, low, high, n = patient_interval(group[metric], BOOTSTRAPS, SEED)
        summary_rows.append(dict(baseline=baseline, comparison=method, metric=metric, mean=mean, ci_low=low, ci_high=high, patients=n))
for baseline in ["zero", "train_mean"]:
    subset = aggregated[aggregated.baseline == baseline]
    for metric in ["kl", "logit_drop", "target_minus_control"]:
        table = subset.pivot(index="pid", columns="method", values=metric)
        for control in ["semantic", "functional", "random"]:
            mean, low, high, n = patient_interval(table.scag - table[control], BOOTSTRAPS, SEED)
            summary_rows.append(dict(baseline=baseline, comparison=f"scag minus {control}", metric=metric, mean=mean, ci_low=low, ci_high=high, patients=n))
summary = pd.DataFrame(summary_rows)
summary.to_csv(INTERVENTION_DIR / "summary.csv", index=False)
display(summary)

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MaxNLocator

OURS = "SENCoG"  # display name only; summary key stays "scag"
methods = ["semantic", "functional", "random", "scag"]
names = ["Semantic", "Functional", "Random", OURS]
baselines = [("zero", "Zeroing", "#4C72B0"), ("train_mean", "Training mean", "#DD8452")]

rc = {"font.family": "serif", "font.serif": ["Times New Roman", "Nimbus Roman", "STIXGeneral", "DejaVu Serif"],
      "mathtext.fontset": "stix", "font.size": 8, "axes.titlesize": 8.5, "pdf.fonttype": 42, "ps.fonttype": 42}
with plt.rc_context(rc):
    fig, axes = plt.subplots(1, 2, figsize=(5.5, 2.1))
    w = .36
    for ax, metric, title in zip(axes, ["kl", "flip"], ["(a) KL divergence", "(b) Prediction flips"]):
        ax.set_axisbelow(True)
        ax.grid(axis="y", color="#E5E5E5", linewidth=.5)
        tops = []
        for j, (baseline, label, color) in enumerate(baselines):
            s = summary[(summary.baseline == baseline) & (summary.metric == metric)].set_index("comparison").reindex(methods)
            m = s["mean"].to_numpy(float)
            err = np.vstack([m - s.ci_low.to_numpy(float), s.ci_high.to_numpy(float) - m])
            err = np.clip(np.where(np.isfinite(err), err, 0), 0, None)
            xs = np.arange(len(methods)) + (j - .5) * w
            ax.bar(xs, m, w, color=color, label=label, linewidth=0, yerr=err,
                   error_kw=dict(ecolor="black", elinewidth=.7, capsize=1.5, capthick=.7), zorder=3)
            for xi, mi in zip(xs, m):
                if mi == 0:
                    ax.text(xi, 0, "0", ha="center", va="bottom", fontsize=6, color="#555555", zorder=4)
            tops.append(np.nanmax(np.where(np.isfinite(m), m + err[1], 0)))
        ax.set(xticks=range(len(methods)), xticklabels=names, xlim=(-.6, len(methods) - .4), title=title)
        ax.get_xticklabels()[-1].set_fontweight("bold")
        ax.tick_params(labelsize=7, length=2.5, width=.6)
        if metric == "flip":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.set_ylim(0, 1)
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
        else:
            ax.yaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
            top = max(tops)
            ax.set_ylim(0, max(ax.yaxis.get_major_locator().tick_values(0, top)[-1], top))
            ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 3), useMathText=True)
        for sp in ["top", "right"]:
            ax.spines[sp].set_visible(False)
        for sp in ["bottom", "left"]:
            ax.spines[sp].set_linewidth(.6)
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", bbox_to_anchor=(.5, 1.03), ncol=2,
               frameon=False, fontsize=7.5, handlelength=1.2, columnspacing=1.5)
    fig.subplots_adjust(left=.08, right=.99, bottom=.12, top=.80, wspace=.3)
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(INTERVENTION_DIR / f"comparison.{ext}", dpi=400, bbox_inches="tight", facecolor="white")
    plt.show()

### One prespecified patient's evidence


In [ ]:
assert 0 <= PATIENT_INDEX < len(eval_ids)
pid = eval_ids[PATIENT_INDEX]
channels = source_sets['scag']
targets, controls = target_sets['scag']
changed_logits, changed_target = intervention_cache[('zero', tuple(sorted(channels.tolist())))]
row = patient_effects[(patient_effects.pid == pid) & (patient_effects.strategy == 'scag') & (patient_effects.baseline == 'zero')].iloc[0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
axes[0].imshow(eval_inputs[PATIENT_INDEX, 0], cmap='gray')
axes[0].set_title(f'{pid}: ED'); axes[0].axis('off')
# Training SD scaling avoids one high-magnitude channel dominating the visual mean.
source_sd = source_train.std(axis=0, ddof=1)
scales = torch.as_tensor(np.maximum(source_sd[channels], 1e-8), dtype=torch.float32)
activation = (ml_acts[ML_EVAL_SPLIT][SOURCE_LAYER][PATIENT_INDEX, channels] / scales[:, None, None]).mean(0)
heat = torch.nn.functional.interpolate(activation[None, None], size=eval_inputs.shape[-2:], mode='bilinear', align_corners=False)[0, 0].numpy()
axes[1].imshow(eval_inputs[PATIENT_INDEX, 0], cmap='gray')
axes[1].imshow(heat, cmap='magma', alpha=.5)
axes[1].set_title(f'{ML_QUERY}-associated source group'); axes[1].axis('off')
if len(targets):
    effect = (clean_target[PATIENT_INDEX, targets] - changed_target[PATIENT_INDEX, targets]) / target_sd[targets]
    axes[2].bar(range(len(targets)), effect)
    axes[2].set(xticks=range(len(targets)), xticklabels=[f'n{i}' for i in targets], ylabel='Signed change / training SD', title='Downstream target responses')
    axes[2].tick_params(axis='x', labelrotation=90)
else:
    axes[2].text(.1, .5, 'No eligible connected targets')
axes[3].bar(['Original', 'Intervened'], [row.probability_before, row.probability_after], color=['#778395', '#dd6d6d'])
axes[3].set(ylim=(0, 1), ylabel='Probability', title=row.original_class)
fig.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(INTERVENTION_DIR / f'patient_{PATIENT_INDEX}_evidence.{ext}', dpi=250, bbox_inches='tight')
plt.show()
statement = f"Patient {pid}: the model predicted {row.original_class} with probability {row.probability_before:.3f}. Zeroing {SOURCE_K} training-selected {ML_QUERY}-associated SCAG channels in {SOURCE_LAYER} changed that probability to {row.probability_after:.3f}. The mean absolute standardised change across {len(targets)} retrieved channels in {TARGET_LAYER} was {row.target_effect:.3f}; the unconnected-control mean was {row.control_effect:.3f}."
limitations = "This is a patient-level evaluation of a globally selected group, not an instance-optimised explanation. Heatmaps use this patient's activations and no evaluation masks; ED is shown as context for a response to stacked ED/ES input. Group interventions do not establish individual edge effects, anatomical causality, or a unique computational pathway. Zeroing and mean replacement may create atypical internal states. Target controls are not activation/degree matched. Bootstrap intervals are pointwise and conditional on this trained model and frozen selections."
(INTERVENTION_DIR / 'RESULTS.md').write_text('# Cross-stage SCAG intervention experiment\n\n' + statement + '\n\n' + limitations + '\n\n```\n' + summary.to_string(index=False) + '\n```\n')
(INTERVENTION_DIR / f'patient_{PATIENT_INDEX}_explanation.json').write_text(json.dumps(dict(pid=pid, source_nodes=[f'{SOURCE_LAYER}:n{i}' for i in channels], target_nodes=[f'{TARGET_LAYER}:n{i}' for i in targets], statement=statement, limitations=limitations), indent=2))
display(Markdown(statement + '\n\n' + limitations))
print('Results:', INTERVENTION_DIR)


## 8. Compare traversal rules and inspect one patient

Run this section after the existing intervention helper and bootstrap cells. The three methods query the same graph with the same final-stage seeds. Each preceding stage uses a common training-feasible budget, without filling missing label-constrained candidates. Interventions act on the exact retrieved groups, one source stage at a time; they are not simultaneous whole-path ablations. Random source controls match the source budget. Previous experiments remain separate.


### Training-only, budget-controlled traversal


In [ ]:
TR_DIR = ML_DIR / 'traversal_comparison' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
TR_DIR.mkdir(parents=True)
TR_METHODS = ['label_constrained', 'unconstrained', 'soft_guided']
TR_NAMES = ['Label-constrained', 'Unconstrained', 'Soft-guided']
assert TR_STAGE_BUDGET > 0 and TR_RANDOM_REPEATS > 0 and all(c in CONCEPTS for c in TR_QUERIES)

def traversal_candidates(cross, parents, scores, labels, query, threshold, method):
    if len(parents) == 0:
        return np.array([], dtype=int), np.zeros(cross.shape[0])
    retained = np.where((cross[:, parents] >= threshold) & (cross[:, parents] > 0), cross[:, parents], 0)
    strength = retained.max(axis=1)
    eligible = strength > 0
    if method == 'label_constrained':
        eligible &= np.asarray(labels) == query
    elif method not in ['unconstrained', 'soft_guided']:
        raise ValueError('Unknown traversal strategy')
    ranking = strength * scores if method == 'soft_guided' else strength
    candidates = np.flatnonzero(eligible)
    return candidates[np.lexsort((candidates, -ranking[candidates]))], ranking

check_cross = np.array([[.9], [.7], [.1]])
check_labels = ['MYO', 'LV', 'LV']
assert traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'unconstrained')[0].tolist() == [0, 1]
assert traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'label_constrained')[0].tolist() == [1]
assert traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'soft_guided')[0].tolist() == [1, 0]
assert len(traversal_candidates(check_cross, [], np.ones(3), check_labels, 'LV', .3, 'soft_guided')[0]) == 0
tr_sets, tr_node_rows, tr_edge_rows, tr_budget_rows = {}, [], [], []
for query in TR_QUERIES:
    c = CONCEPTS.index(query)
    final_labels = ml_labels[ML_LAYERS[-1]]
    seeds = final_labels[final_labels.label == query].sort_values(['top_iou', 'neuron'], ascending=[False, True]).head(TR_STAGE_BUDGET).neuron.to_numpy(dtype=int)
    if not len(seeds):
        raise ValueError(f'No final-stage seeds for {query}')
    for method in TR_METHODS:
        tr_sets[(query, method)] = {ML_LAYERS[-1]: seeds.copy()}
    for lower, upper in reversed(list(zip(ML_LAYERS[:-1], ML_LAYERS[1:]))):
        rankings = {}
        cross_matrix = ml_cross[('train', lower, upper)]
        for method in TR_METHODS:
            rankings[method], _ = traversal_candidates(cross_matrix, tr_sets[(query, method)][upper], ml_scores[('train', lower)][:, c], ml_labels[lower].label.to_numpy(), query, TAU, method)
        common_k = min(TR_STAGE_BUDGET, *(len(v) for v in rankings.values()))
        for method in TR_METHODS:
            tr_sets[(query, method)][lower] = rankings[method][:common_k]
            tr_budget_rows.append(dict(query=query, method=method, layer=lower, eligible=len(rankings[method]), selected=common_k, shortfall=TR_STAGE_BUDGET-common_k))
    for method in TR_METHODS:
        for layer in ML_LAYERS:
            selected = tr_sets[(query, method)][layer]
            for i in selected:
                label = ml_labels[layer].iloc[i]
                tr_node_rows.append(dict(query=query, method=method, layer=layer, neuron=int(i), node=f'{layer}:n{i}', label=label.label, train_query_iou=float(ml_scores[('train', layer)][i, c]), tie=bool(label.tie), all_zero=bool(label.all_zero)))
        for lower, upper in zip(ML_LAYERS[:-1], ML_LAYERS[1:]):
            for i in tr_sets[(query, method)][lower]:
                for j in tr_sets[(query, method)][upper]:
                    r = float(ml_cross[('train', lower, upper)][i,j])
                    if r >= TAU and r > 0:
                        tr_edge_rows.append(dict(query=query, method=method, source=f'{lower}:n{i}', target=f'{upper}:n{j}', train_r=r, eval_r=float(ml_cross[(ML_EVAL_SPLIT, lower, upper)][i,j])))
    for layer in ML_LAYERS:
        assert len({len(tr_sets[(query, m)][layer]) for m in TR_METHODS}) == 1
tr_nodes = pd.DataFrame(tr_node_rows)
tr_edges = pd.DataFrame(tr_edge_rows, columns=['query', 'method', 'source', 'target', 'train_r', 'eval_r'])
tr_budgets = pd.DataFrame(tr_budget_rows)
tr_nodes.to_csv(TR_DIR / 'selected_nodes.csv', index=False)
tr_edges.to_csv(TR_DIR / 'candidate_edges.csv', index=False)
tr_budgets.to_csv(TR_DIR / 'budget_audit.csv', index=False)
(TR_DIR / 'protocol.json').write_text(json.dumps(dict(queries=TR_QUERIES, layers=ML_LAYERS, budget=TR_STAGE_BUDGET, seed=SEED, threshold=TAU, random_repeats=TR_RANDOM_REPEATS, patient_index=TR_PATIENT_INDEX, split=ML_EVAL_SPLIT, checkpoint_sha256=checkpoint_sha256, budget_rule='At each stage, all strategies use the minimum available candidate count capped at the prespecified budget. Zero stops all strategies at that stage.', ranking='Maximum retained correlation to any selected parent; soft multiplies by training query IoU.', source_run=str(ML_DIR)), indent=2))
display(tr_budgets)


### Layered graphs for the three traversal strategies

Same coordinates, colours and selected stage budgets across methods. These views display the training-selected correlation graph; intervention effects are reported separately. No model inference is needed.


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Rectangle

TR_PALETTE = {'RV': '#2d6cab', 'MYO': '#3bab3b', 'LV': '#dd6d6d'}
TR_BACKGROUND_MAX = 80
tr_graph_labels = ml_nodes.set_index('node').label.to_dict()

def draw_traversal_graph(ax, query, method, positions, context, display_edges, perspective=False):
    selected = {f'{layer}:n{i}' for layer in ML_LAYERS for i in tr_sets[(query, method)][layer]}
    seeds = {f'{ML_LAYERS[-1]}:n{i}' for i in tr_sets[(query, method)][ML_LAYERS[-1]]}
    xx, yy = np.meshgrid([-1.04, 1.04], [-.79, .79])
    for stage, layer in enumerate(ML_LAYERS):
        z = 1.45 * stage
        if perspective:
            ax.plot_surface(xx, yy, np.full_like(xx, z), color='#DDE5ED', alpha=.10, shade=False, linewidth=0)
        else:
            ax.add_patch(Rectangle((-1.02, z-.19), 2.04, .38, facecolor='#EEF2F6', edgecolor='#DDE3EB', lw=.5, zorder=0))
            ax.text(-1.10, z, f'Stage {stage+1}', ha='right', va='center', fontsize=7, color='#273547')
        if context[layer]:
            pts = np.array([positions[n] for n in context[layer]])
            colors = [TR_PALETTE[tr_graph_labels[n]] for n in context[layer]]
            if perspective:
                ax.scatter(*pts.T, c=colors, s=4, alpha=.2, linewidths=0, depthshade=False, rasterized=True)
            else:
                ax.scatter(pts[:,0], pts[:,2]+.18*pts[:,1], c=colors, s=3.5, alpha=.22, linewidths=0, zorder=1, rasterized=True)
    for a,b in sorted(display_edges):
        p,q = positions[a],positions[b]
        edge = ml_graph[a][b]
        retained = np.isfinite(edge['eval_r']) and edge['eval_r'] >= TAU
        same = a.split(':n')[0] == b.split(':n')[0]
        style = dict(color=('#8795A5' if same else '#273547') if retained else '#B3BBC6', linewidth=.56+.85*np.clip(edge['weight'],0,1), linestyle='-' if retained else (0,(3,2)), alpha=.75)
        if perspective:
            ax.plot([p[0],q[0]],[p[1],q[1]],[p[2],q[2]],**style)
        elif same:
            ax.add_patch(FancyArrowPatch((p[0],p[2]+.18*p[1]),(q[0],q[2]+.18*q[1]),arrowstyle='-',connectionstyle='arc3,rad=.33',zorder=2,**style))
        else:
            ax.plot([p[0],q[0]],[p[2]+.18*p[1],q[2]+.18*q[1]],zorder=2,**style)
    for seed in [False,True]:
        nodes = sorted(n for n in selected if (n in seeds)==seed)
        if not nodes:continue
        pts = np.array([positions[n] for n in nodes])
        kwargs = dict(c=[TR_PALETTE[tr_graph_labels[n]] for n in nodes],s=45 if seed else 29,edgecolors='#273547' if seed else 'white',linewidths=.85)
        if perspective:ax.scatter(*pts.T,depthshade=False,**kwargs)
        else:ax.scatter(pts[:,0],pts[:,2]+.18*pts[:,1],zorder=5,**kwargs)
    if perspective:
        ax.view_init(elev=21,azim=-61)
        ax.set_proj_type('ortho')
        ax.set(xlim=(-1.06,1.06),ylim=(-.88,.88),zlim=(-.12,1.45*(len(ML_LAYERS)-1)+.12))
        ax.set_box_aspect((1.35,1,2.25),zoom=1.05)
    else:
        ax.set(xlim=(-1.65,1.10),ylim=(-.38,1.45*(len(ML_LAYERS)-1)+.42))
    ax.set_axis_off()

tr_display_rows, tr_layout_rows = [], []
with plt.rc_context({'font.family':'DejaVu Sans','font.size':8,'axes.titlesize':9,'pdf.fonttype':42,'ps.fonttype':42}):
    for query in TR_QUERIES:
        selected_union = {f'{layer}:n{i}' for method in TR_METHODS for layer in ML_LAYERS for i in tr_sets[(query,method)][layer]}
        edge_sets = {}
        for method in TR_METHODS:
            part = tr_edges[(tr_edges['query']==query)&(tr_edges.method==method)]
            edges = {tuple(sorted((r.source,r.target))) for r in part.itertuples()}
            # Same display rule as the original: one strongest within-stage neighbour per selected channel.
            for layer in ML_LAYERS:
                channels = tr_sets[(query,method)][layer]
                for i in channels:
                    node = f'{layer}:n{i}'
                    neighbors = [f'{layer}:n{j}' for j in channels if i!=j and ml_graph.has_edge(node,f'{layer}:n{j}')]
                    if neighbors:
                        neighbor = min(neighbors,key=lambda n:(-ml_graph[node][n]['weight'],n))
                        edges.add(tuple(sorted((node,neighbor))))
            selected = {f'{layer}:n{i}' for layer in ML_LAYERS for i in tr_sets[(query,method)][layer]}
            assert all(a in selected and b in selected and ml_graph.has_edge(a,b) for a,b in edges)
            edge_sets[method] = edges
            for a,b in sorted(edges):tr_display_rows.append(dict(query=query,method=method,source=a,target=b,**ml_graph[a][b]))
        union_edges = set().union(*edge_sets.values())
        positions,context,orders = {},{},{}
        for stage,layer in enumerate(ML_LAYERS):
            ids = sorted(ml_nodes.loc[ml_nodes.layer==layer,'node'])
            for k,node in enumerate(ids):
                radius = np.sqrt((k+.5)/max(len(ids),1));angle = k*np.pi*(3-np.sqrt(5))
                positions[node] = (.98*radius*np.cos(angle),.72*radius*np.sin(angle),1.45*stage)
            others = [n for n in ids if n not in selected_union]
            if len(others)>TR_BACKGROUND_MAX:
                keep = np.sort(np.random.default_rng(SEED+7919*(stage+1)).choice(len(others),TR_BACKGROUND_MAX,replace=False))
                others = [others[i] for i in keep]
            context[layer] = others
        orders[ML_LAYERS[-1]] = [f'{ML_LAYERS[-1]}:n{i}' for i in tr_sets[(query,TR_METHODS[0])][ML_LAYERS[-1]]]
        for lower,upper in reversed(list(zip(ML_LAYERS[:-1],ML_LAYERS[1:]))):
            def barycentre(node):
                ranks = [j for j,n in enumerate(orders[upper]) if tuple(sorted((node,n))) in union_edges]
                return np.mean(ranks) if ranks else float('inf')
            orders[lower] = sorted([n for n in selected_union if n.split(':n')[0]==lower],key=lambda n:(barycentre(n),n))
        for stage,layer in enumerate(ML_LAYERS):
            ordered = orders[layer]
            for j,(node,x) in enumerate(zip(ordered,np.linspace(-.82,.82,len(ordered)) if len(ordered)>1 else [0.])):
                positions[node] = (float(x),float(.21*np.sin(j*2.39996+stage)),1.45*stage)
        for node,(x,y,z) in positions.items():tr_layout_rows.append(dict(query=query,node=node,x=x,y=y,z=z,context=any(node in v for v in context.values())))
        handles = [Line2D([],[],marker='o',linestyle='None',markerfacecolor=color,markeredgecolor='white',label=c) for c,color in TR_PALETTE.items()]
        handles.append(Line2D([],[],marker='o',linestyle='None',markerfacecolor=TR_PALETTE[query],markeredgecolor='#273547',label=f'{query} seeds'))
        fig = plt.figure(figsize=(10.5,7))
        for col,(method,name) in enumerate(zip(TR_METHODS,TR_NAMES)):
            ax = fig.add_subplot(2,3,col+1)
            draw_traversal_graph(ax,query,method,positions,context,edge_sets[method])
            ax.set_title(f'({chr(97+col)}) {name}',fontweight='bold')
            ax = fig.add_subplot(2,3,col+4,projection='3d')
            draw_traversal_graph(ax,query,method,positions,context,edge_sets[method],perspective=True)
        fig.legend(handles=handles,loc='lower center',ncol=4,frameon=False)
        fig.subplots_adjust(left=.025,right=.99,top=.94,bottom=.065,wspace=.13,hspace=.05)
        for ext in ['png','pdf','svg']:fig.savefig(TR_DIR/f'{query}_traversal_graphs.{ext}',dpi=350,bbox_inches='tight',facecolor='white')
        plt.show()
        for method,name in zip(TR_METHODS,TR_NAMES):
            fig = plt.figure(figsize=(7.4,4.45))
            ax2 = fig.add_axes([.03,.17,.48,.75]);ax3 = fig.add_axes([.50,.17,.47,.75],projection='3d')
            draw_traversal_graph(ax2,query,method,positions,context,edge_sets[method])
            draw_traversal_graph(ax3,query,method,positions,context,edge_sets[method],perspective=True)
            ax2.set_title(f'{query} · {name}',loc='left',fontweight='bold')
            ax3.set_title('Layered view',loc='left',fontweight='bold')
            fig.legend(handles=handles,loc='lower center',ncol=4,frameon=False)
            for ext in ['png','pdf','svg']:fig.savefig(TR_DIR/f'{query}_{method}_layered.{ext}',dpi=350,bbox_inches='tight',facecolor='white')
            plt.close(fig)
pd.DataFrame(tr_display_rows).to_csv(TR_DIR/'graph_display_edges.csv',index=False)
pd.DataFrame(tr_layout_rows).to_csv(TR_DIR/'graph_layout.csv',index=False)
(TR_DIR/'GRAPH_CAPTION.md').write_text('Concept-guided cross-stage SCAG neighbourhoods retrieved using label-constrained, unconstrained and soft-guided traversal. Columns use the same node coordinates and stage budgets; colours indicate assigned concepts and outlined nodes are final-stage seeds. Upper and lower rows show planar and perspective views.\n\nDisplay notes: all retrieved adjacent-stage edges are shown, with one strongest within-stage neighbour per selected channel as context. Background nodes are sampled from channels outside the union of selected sets. Solid/dashed lines indicate evaluation correlations above/below the training threshold. Coordinates are schematic; edges remain statistical associations, not intervention-measured dependencies.\n')
print('Layered comparison figures:',TR_DIR)


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.path import Path
from matplotlib.patches import PathPatch

OURS = "SENCoG"      # display name only
STAGE_TAG = "Block"  # column label prefix
TR_PALETTE = {'RV': '#2d6cab', 'MYO': '#3bab3b', 'LV': '#dd6d6d'}
TR_BACKGROUND_MAX = 80
TR_RC = {'font.family': 'serif', 'font.serif': ['Times New Roman', 'Nimbus Roman', 'STIXGeneral', 'DejaVu Serif'],
         'mathtext.fontset': 'stix', 'font.size': 7, 'pdf.fonttype': 42, 'ps.fonttype': 42}
INK, GREY, CAPSULE, POOL = '#2B3440', '#9AA5B1', '#F3F5F8', '#BFC6CE'
DASH = (0, (4, 3))
tr_graph_labels = ml_nodes.set_index('node').label.to_dict()
stage_of = {l: i for i, l in enumerate(ML_LAYERS)}
N_ST = len(ML_LAYERS)
X = lambda stage: N_ST - 1 - stage   # last block is drawn leftmost

def _curve(p, q, bulge=None):
    (x0, y0), (x1, y1) = p, q
    if bulge is None:
        xm = (x0 + x1) / 2
        pts = [(x0, y0), (xm, y0), (xm, y1), (x1, y1)]
    else:
        pts = [(x0, y0), (x0 + bulge, y0), (x1 + bulge, y1), (x1, y1)]
    return Path(pts, [Path.MOVETO] + [Path.CURVE4] * 3)

def draw_flow(ax, query, method, positions, context, display_edges):
    selected = {f'{l}:n{i}' for l in ML_LAYERS for i in tr_sets[(query, method)][l]}
    seeds = {f'{ML_LAYERS[-1]}:n{i}' for i in tr_sets[(query, method)][ML_LAYERS[-1]]}
    xy = lambda n: (X(stage_of[n.split(':n')[0]]), positions[n][0])
    for s, layer in enumerate(ML_LAYERS):
        x = X(s)
        ax.plot([x, x], [-1.0, 1.0], lw=15, color=CAPSULE, solid_capstyle='round', zorder=0)   # column capsule
        ax.text(x, -1.16, f'{STAGE_TAG} {s+1}', ha='center', va='top', fontsize=6.5, color=INK)
        if context[layer]:                                                                      # unselected channel pool
            pts = np.array([positions[n] for n in context[layer]])
            ax.scatter(x + .15 * pts[:, 1] / .72, pts[:, 0], s=2, color=POOL, linewidths=0, zorder=1)
    items = []
    for a, b in display_edges:
        if stage_of[a.split(':n')[0]] > stage_of[b.split(':n')[0]]:   # b = endpoint in the later block
            a, b = b, a
        e = ml_graph[a][b]
        retained = bool(np.isfinite(e['eval_r']) and e['eval_r'] >= TAU)
        same = a.split(':n')[0] == b.split(':n')[0]
        items.append((retained, same, a, b, xy(a), xy(b), np.clip(e['weight'], 0, 1)))
    for retained, same, a, b, p, q, w in sorted(items, key=lambda t: (t[0], not t[1])):
        if same:
            path = _curve(p, q, bulge=min(.2 + .6 * abs(q[1] - p[1]), .55))
            color, lw, alpha = GREY, .35 + .4 * w, .7
        else:
            path = _curve(p, q)
            color, lw, alpha = TR_PALETTE[tr_graph_labels[b]], .35 + .9 * w, .6
        ax.add_patch(PathPatch(path, fc='none', ec=color, lw=lw, ls='-' if retained else DASH,
                               alpha=alpha if retained else alpha * .55, capstyle='round', zorder=2))
    for seed in [False, True]:
        nodes = sorted(n for n in selected if (n in seeds) == seed)
        if not nodes:
            continue
        pts = np.array([xy(n) for n in nodes])
        ax.scatter(pts[:, 0], pts[:, 1], c=[TR_PALETTE[tr_graph_labels[n]] for n in nodes], s=34 if seed else 17,
                   edgecolors=INK if seed else 'white', linewidths=.7 if seed else .4, zorder=6 if seed else 5)
    ax.annotate('', xy=(N_ST - 1, 1.2), xytext=(0, 1.2), zorder=1,
                arrowprops=dict(arrowstyle='-|>', lw=.6, color=GREY, mutation_scale=6, shrinkA=0, shrinkB=0))
    ax.text((N_ST - 1) / 2, 1.25, 'traversal from seeds', ha='center', va='bottom', fontsize=6, color=GREY, style='italic')
    ax.set(xlim=(-.45, N_ST - 1 + .6), ylim=(-1.3, 1.45))
    ax.set_axis_off()

tr_display_rows, tr_layout_rows = [], []
with plt.rc_context(TR_RC):
    for query in TR_QUERIES:
        selected_union = {f'{layer}:n{i}' for method in TR_METHODS for layer in ML_LAYERS for i in tr_sets[(query,method)][layer]}
        edge_sets = {}
        for method in TR_METHODS:
            part = tr_edges[(tr_edges['query']==query)&(tr_edges.method==method)]
            edges = {tuple(sorted((r.source,r.target))) for r in part.itertuples()}
            # Same display rule as the original: one strongest within-stage neighbour per selected channel.
            for layer in ML_LAYERS:
                channels = tr_sets[(query,method)][layer]
                for i in channels:
                    node = f'{layer}:n{i}'
                    neighbors = [f'{layer}:n{j}' for j in channels if i!=j and ml_graph.has_edge(node,f'{layer}:n{j}')]
                    if neighbors:
                        neighbor = min(neighbors,key=lambda n:(-ml_graph[node][n]['weight'],n))
                        edges.add(tuple(sorted((node,neighbor))))
            selected = {f'{layer}:n{i}' for layer in ML_LAYERS for i in tr_sets[(query,method)][layer]}
            assert all(a in selected and b in selected and ml_graph.has_edge(a,b) for a,b in edges)
            edge_sets[method] = edges
            for a,b in sorted(edges):tr_display_rows.append(dict(query=query,method=method,source=a,target=b,**ml_graph[a][b]))
        union_edges = set().union(*edge_sets.values())
        positions,context,orders = {},{},{}
        for stage,layer in enumerate(ML_LAYERS):
            ids = sorted(ml_nodes.loc[ml_nodes.layer==layer,'node'])
            for k,node in enumerate(ids):
                radius = np.sqrt((k+.5)/max(len(ids),1));angle = k*np.pi*(3-np.sqrt(5))
                positions[node] = (.98*radius*np.cos(angle),.72*radius*np.sin(angle),1.45*stage)
            others = [n for n in ids if n not in selected_union]
            if len(others)>TR_BACKGROUND_MAX:
                keep = np.sort(np.random.default_rng(SEED+7919*(stage+1)).choice(len(others),TR_BACKGROUND_MAX,replace=False))
                others = [others[i] for i in keep]
            context[layer] = others
        orders[ML_LAYERS[-1]] = [f'{ML_LAYERS[-1]}:n{i}' for i in tr_sets[(query,TR_METHODS[0])][ML_LAYERS[-1]]]
        for lower,upper in reversed(list(zip(ML_LAYERS[:-1],ML_LAYERS[1:]))):
            def barycentre(node):
                ranks = [j for j,n in enumerate(orders[upper]) if tuple(sorted((node,n))) in union_edges]
                return np.mean(ranks) if ranks else float('inf')
            orders[lower] = sorted([n for n in selected_union if n.split(':n')[0]==lower],key=lambda n:(barycentre(n),n))
        for stage,layer in enumerate(ML_LAYERS):
            ordered = orders[layer]
            for j,(node,x) in enumerate(zip(ordered,np.linspace(-.82,.82,len(ordered)) if len(ordered)>1 else [0.])):
                positions[node] = (float(x),float(.21*np.sin(j*2.39996+stage)),1.45*stage)
        for node,(x,y,z) in positions.items():tr_layout_rows.append(dict(query=query,node=node,x=x,y=y,z=z,context=any(node in v for v in context.values())))

        handles = [Line2D([], [], marker='o', ls='None', markersize=4, markerfacecolor=c, markeredgecolor='white',
                          markeredgewidth=.4, label=k) for k, c in TR_PALETTE.items()]
        handles += [Line2D([], [], marker='o', ls='None', markersize=5.5, markerfacecolor=TR_PALETTE[query],
                           markeredgecolor=INK, markeredgewidth=.7, label=f'{query} seeds'),
                    # Line2D([], [], color=INK, lw=.9, label=r'$r\geq\tau$'),
                    # Line2D([], [], color=INK, lw=.9, ls=DASH, label=r'$r<\tau$')
                    ]
        legend_kw = dict(handles=handles, loc='upper center', frameon=False, fontsize=7, handletextpad=.2, columnspacing=1.1)

        fig = plt.figure(figsize=(6.9, 2.8))
        gs = fig.add_gridspec(1, 3, left=.005, right=.995, top=.88, bottom=.1, wspace=.03)
        for col, (method, name) in enumerate(zip(TR_METHODS, TR_NAMES)):
            ax = fig.add_subplot(gs[0, col])
            draw_flow(ax, query, method, positions, context, edge_sets[method])
            ax.text(.5, -.01, f'({chr(97+col)}) {name}', transform=ax.transAxes, ha='center', va='top', fontsize=8)
        fig.legend(bbox_to_anchor=(.5, 1.01), ncol=6, **legend_kw)
        for ext in ['png', 'pdf', 'svg']:
            fig.savefig(TR_DIR/f'{query}_traversal_graphs.{ext}', dpi=300, bbox_inches='tight', pad_inches=.03, facecolor='white')
        plt.show()

        for method, name in zip(TR_METHODS, TR_NAMES):
            fig = plt.figure(figsize=(3.4, 3.0))
            ax = fig.add_axes([.01, .12, .98, .70])
            draw_flow(ax, query, method, positions, context, edge_sets[method])
            ax.text(.5, -.01, f'{query} · {name}', transform=ax.transAxes, ha='center', va='top', fontsize=8)
            fig.legend(bbox_to_anchor=(.5, 1.01), ncol=3, **legend_kw)
            for ext in ['png', 'pdf', 'svg']:
                fig.savefig(TR_DIR/f'{query}_{method}_flow.{ext}', dpi=300, bbox_inches='tight', pad_inches=.03, facecolor='white')
            plt.close(fig)

pd.DataFrame(tr_display_rows).to_csv(TR_DIR/'graph_display_edges.csv',index=False)
pd.DataFrame(tr_layout_rows).to_csv(TR_DIR/'graph_layout.csv',index=False)
(TR_DIR/'GRAPH_CAPTION.md').write_text(f'Concept-guided cross-stage {OURS} neighbourhoods retrieved using label-constrained, unconstrained and soft-guided traversal. Blocks are ordered right to left from Block 1 to Block {N_ST}, so the final-block seeds sit leftmost and traversal proceeds rightwards to earlier blocks; columns share node coordinates and stage budgets. Node colours indicate assigned concepts and outlined nodes are final-block seeds. Cross-stage edges take the colour of their endpoint in the later block; grey arcs are within-block neighbours. Solid/dashed lines indicate evaluation correlations above/below the training threshold; grey dots are unselected channels.\n\nDisplay notes: all retrieved adjacent-stage edges are shown, with one strongest within-stage neighbour per selected channel as context. Background nodes are sampled from channels outside the union of selected sets. Coordinates are schematic; edges remain statistical associations, not intervention-measured dependencies.\n')
print('Flow figures:',TR_DIR)

### Same patient's ED/ES responses across all stages


In [ ]:
assert 0 <= TR_PATIENT_INDEX < len(images[ML_EVAL_SPLIT])
tr_pid = str(manifests[ML_EVAL_SPLIT].iloc[TR_PATIENT_INDEX].pid)
tr_heat_rows = []
for query in TR_QUERIES:
    fig, axes = plt.subplots(3, len(ML_LAYERS), figsize=(12, 7), squeeze=False)
    for row, (method, method_name) in enumerate(zip(TR_METHODS, TR_NAMES)):
        for col, layer in enumerate(ML_LAYERS):
            ax = axes[row, col]
            selected = tr_sets[(query, method)][layer]
            if not len(selected):
                ax.text(.5, .5, 'No eligible group', ha='center'); ax.axis('off'); continue
            train_maps = ml_acts['train'][layer]
            scales = train_maps.mean((2,3)).std(dim=0).numpy()
            valid = selected[scales[selected] > 1e-8]
            if not len(valid):
                ax.text(.5, .5, 'Constant channels', ha='center'); ax.axis('off'); continue
            # Common training-derived per-channel calibration; display only, never used for selection.
            group = (ml_acts[ML_EVAL_SPLIT][layer][TR_PATIENT_INDEX, valid] / torch.as_tensor(scales[valid])[:,None,None]).mean(0)
            reference = (train_maps[:,valid] / torch.as_tensor(scales[valid])[None,:,None,None]).mean(1).numpy()
            vmax = max(float(np.percentile(reference, 99)), 1e-8)
            heat = torch.nn.functional.interpolate(group[None,None], size=images[ML_EVAL_SPLIT].shape[-2:], mode='bilinear', align_corners=False)[0,0].numpy()
            ed, es = images[ML_EVAL_SPLIT][TR_PATIENT_INDEX].numpy()
            paired = np.concatenate([ed, es], axis=1)
            ax.imshow(paired, cmap='gray')
            ax.imshow(np.concatenate([heat,heat], axis=1), cmap='magma', vmin=0, vmax=vmax, alpha=.45)
            ax.set_title(f'{layer.split(".")[-1]} · {len(selected)} channels', fontsize=9)
            ax.set_xlabel('ED | ES', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:ax.set_ylabel(method_name, fontsize=9)
            tr_heat_rows.append(dict(query=query, method=method, layer=layer, pid=tr_pid, channels=selected.tolist(), displayed_channels=valid.tolist(), display_vmax=vmax))
    fig.suptitle(f'{query} · patient {tr_pid} · joint ED/ES activations', fontsize=11)
    fig.tight_layout()
    for ext in ['png','pdf']:
        fig.savefig(TR_DIR / f'{query}_patient_{TR_PATIENT_INDEX}_stages.{ext}', dpi=250, bbox_inches='tight')
    plt.show()
(TR_DIR / 'patient_view_provenance.json').write_text(json.dumps(tr_heat_rows, indent=2))


### Intervene on the exact retrieved groups


In [ ]:
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == checkpoint_sha256
model.eval()
tr_inputs = images[ML_EVAL_SPLIT]
tr_ids = manifests[ML_EVAL_SPLIT].pid.astype(str).tolist()
tr_effects, tr_details, tr_skipped = [], [], []
tr_rng = np.random.default_rng(SEED)
for lower, upper in zip(ML_LAYERS[:-1], ML_LAYERS[1:]):
    means = ml_acts['train'][lower].mean((0,2,3)).numpy()
    target_scale = ml_acts['train'][upper].mean((2,3)).numpy().std(axis=0, ddof=1)
    clean, target_clean = run_stage_intervention(model, tr_inputs, lower, upper, [], 'zero', BATCH_SIZE, DEVICE)
    np.testing.assert_allclose(target_clean, ml_acts[ML_EVAL_SPLIT][upper].mean((2,3)).numpy(), rtol=1e-4, atol=1e-5)
    predicted = clean.argmax(1)
    clean_logp = clean.log_softmax(1)
    idx = torch.arange(len(tr_ids))
    cache = {}
    for query in TR_QUERIES:
        k = len(tr_sets[(query, TR_METHODS[0])][lower])
        if not k:
            tr_skipped.append(dict(query=query, source_layer=lower, reason='No common feasible source budget'))
            continue
        random_groups = [np.sort(tr_rng.choice(len(means), k, replace=False)) for _ in range(TR_RANDOM_REPEATS)]
        for baseline in ['zero','train_mean']:
            for method in TR_METHODS:
                sources = tr_sets[(query,method)][lower]
                targets = tr_sets[(query,method)][upper]
                valid_targets = targets[target_scale[targets] > 1e-8]
                groups = [('retrieved',0,sources)] + [('random',r,g) for r,g in enumerate(random_groups)]
                for selection, repeat, group in groups:
                    key = (baseline,tuple(sorted(group.tolist())))
                    if key not in cache:
                        cache[key] = run_stage_intervention(model, tr_inputs, lower, upper, group, baseline, BATCH_SIZE, DEVICE, source_means=means)
                    changed, target_after = cache[key]
                    kl = (clean_logp.exp()*(clean_logp-changed.log_softmax(1))).sum(1).clamp_min(0).numpy()
                    delta = (target_clean[:,valid_targets]-target_after[:,valid_targets])/target_scale[valid_targets][None,:]
                    target_effect = np.abs(delta).mean(1) if len(valid_targets) else np.full(len(tr_ids),np.nan)
                    c = CONCEPTS.index(query)
                    evidence = ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,c].mean(1)
                    other = np.delete(ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,:],c,axis=2).max(2)
                    selectivity = (ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,c]-other).mean(1)
                    after_prob = changed.softmax(1)[idx,predicted].numpy()
                    for patient,pid in enumerate(tr_ids):
                        tr_effects.append(dict(query=query, method=method, source_layer=lower, target_layer=upper, baseline=baseline, selection=selection, repeat=repeat, pid=pid, source_k=len(group), target_k=len(valid_targets), kl=float(kl[patient]), flip=float(changed[patient].argmax()!=predicted[patient]), target_effect=float(target_effect[patient]), query_iou=float(evidence[patient]), selectivity=float(selectivity[patient]), probability_before=float(clean.softmax(1)[patient,predicted[patient]]), probability_after=float(after_prob[patient]), predicted_class=list(cfg.classes)[int(predicted[patient])], sources=json.dumps(group.tolist()), targets=json.dumps(valid_targets.tolist())))
                        if selection=='retrieved':
                            for pos,j in enumerate(valid_targets):
                                tr_details.append(dict(query=query,method=method,baseline=baseline,pid=pid,source_layer=lower,target_layer=upper,target=int(j),signed_change_sd=float(delta[patient,pos])))
            print(query, lower, '→', upper, baseline, flush=True)
    restored,_ = run_stage_intervention(model,tr_inputs,lower,upper,[],'zero',BATCH_SIZE,DEVICE)
    torch.testing.assert_close(restored,clean)
    pd.DataFrame(tr_effects).to_csv(TR_DIR/'patient_effects.csv',index=False)
tr_results = pd.DataFrame(tr_effects)
pd.DataFrame(tr_details).to_csv(TR_DIR/'downstream_channel_effects.csv',index=False)
(TR_DIR/'skipped.json').write_text(json.dumps(tr_skipped,indent=2))
if tr_results.empty:
    raise RuntimeError('No feasible comparisons. See budget_audit.csv; do not silently relax eligibility.')


### Paired patient comparisons and compact figures


In [ ]:
tr_metrics = ['kl','flip','target_effect','query_iou','selectivity']
tr_keys = ['query','method','source_layer','target_layer','baseline','selection','pid']
tr_patient = tr_results.groupby(tr_keys,as_index=False)[tr_metrics].mean()
tr_summary_rows = []
for keys,group in tr_patient.groupby(tr_keys[:-1]):
    for metric in tr_metrics:
        mean,lo,hi,n = patient_interval(group[metric],BOOTSTRAPS,SEED)
        tr_summary_rows.append(dict(zip(tr_keys[:-1],keys),metric=metric,mean=mean,ci_low=lo,ci_high=hi,patients=n))
tr_summary = pd.DataFrame(tr_summary_rows)
tr_summary.to_csv(TR_DIR/'summary.csv',index=False)
tr_paired_rows = []
for keys,group in tr_patient.groupby(['query','source_layer','baseline']):
    for metric in tr_metrics:
        table = group.pivot(index='pid',columns=['method','selection'],values=metric)
        comparisons = [(f'{m} minus random',table[(m,'retrieved')]-table[(m,'random')]) for m in TR_METHODS]
        comparisons += [(f'{a} minus {b}',table[(a,'retrieved')]-table[(b,'retrieved')]) for a,b in [('soft_guided','unconstrained'),('label_constrained','unconstrained'),('soft_guided','label_constrained')]]
        for comparison,values in comparisons:
            mean,lo,hi,n = patient_interval(values,BOOTSTRAPS,SEED)
            tr_paired_rows.append(dict(query=keys[0],source_layer=keys[1],baseline=keys[2],comparison=comparison,metric=metric,mean=mean,ci_low=lo,ci_high=hi,patients=n))
tr_paired = pd.DataFrame(tr_paired_rows)
tr_paired.to_csv(TR_DIR/'paired_comparisons.csv',index=False)
for query in TR_QUERIES:
    for layer in tr_summary.source_layer.unique():
        fig,axes = plt.subplots(1,2,figsize=(6.2,2.8))
        for ax,metric,title in zip(axes,['kl','flip'],['KL divergence','Prediction flips']):
            for baseline,offset,marker in [('zero',-.1,'o'),('train_mean',.1,'s')]:
                part = tr_summary[(tr_summary['query']==query)&(tr_summary.source_layer==layer)&(tr_summary.baseline==baseline)&(tr_summary.selection=='retrieved')&(tr_summary.metric==metric)].set_index('method').reindex(TR_METHODS)
                x = np.arange(3)+offset
                ax.vlines(x,part.ci_low,part.ci_high,color='#218A78' if baseline=='zero' else '#6486B5')
                ax.plot(x,part['mean'],marker,linestyle='None',label='Zeroing' if baseline=='zero' else 'Training mean')
            ax.set(xticks=range(3),xticklabels=['Constrained','Unconstrained','Soft-guided'],title=title)
            ax.tick_params(axis='x',labelsize=8,rotation=15)
            ax.spines[['top','right']].set_visible(False)
            if metric=='flip':ax.set_ylim(-.025,1.025);ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f'{v:.0%}'))
        axes[0].legend(frameon=False,fontsize=7)
        fig.suptitle(f'{query} · intervention at {layer.split(".")[-1]}',fontsize=10)
        fig.tight_layout()
        for ext in ['png','pdf','svg']:fig.savefig(TR_DIR/f'{query}_{layer}_comparison.{ext}',dpi=300,bbox_inches='tight')
        plt.show()
display(tr_paired)
tr_local = tr_results[(tr_results.pid==tr_pid)&(tr_results.selection=='retrieved')]
statements = []
for r in tr_local.itertuples():
    statements.append(f'{r.query}, {r.method}, {r.source_layer} → {r.target_layer}, {r.baseline}: intervening on {r.source_k} channels changed the original {r.predicted_class} probability from {r.probability_before:.3f} to {r.probability_after:.3f}; KL={r.kl:.5f}, flip={bool(r.flip)}, mean absolute downstream change={r.target_effect:.3f} training SD.')
notes = 'Selection is global and training-derived; the displayed input is fixed across stages. Evaluation masks assess overlap only. Labels are forced argmax assignments, not verified detectors. Heatmap colour scales are training-calibrated per group and not directly comparable across groups. Downstream targets differ between strategies, so target-effect contrasts describe different retrieved groups, not a common target. Random source controls use the same downstream targets for each strategy. No intervention validates an individual correlation edge. Intervals are exploratory, pointwise patient-bootstrap intervals conditional on this model and selection; no multiple-testing correction. Zero and mean replacement can create atypical internal states.'
(TR_DIR/'RESULTS.md').write_text('# Budget-controlled traversal comparison\n\n'+notes+'\n\n## Prespecified patient '+tr_pid+'\n\n'+'\n\n'.join(statements)+'\n\n## Paired comparisons\n\n```\n'+tr_paired.to_string(index=False)+'\n```\n')
print('Traversal results:',TR_DIR)


## 9. Variable-size concept tracing (separate experiment)
The fixed-budget experiment above is unchanged. Use the same training graph, concepts, final-stage seeds, thresholds and intervention baselines. Here each selected parent retrieves up to `VR_NEIGHBORS_PER_PARENT` eligible preceding-stage channels, then duplicates are removed. There is **no per-stage cap or cross-method budget matching**. Label-constrained branches may stop independently; soft guidance ranks correlation × query IoU and unconstrained ranks correlation. A shared positive-correlation eligibility threshold remains in force.

Variable size describes graph expansion. KL/flip differences between methods are confounded by channel count; compare each retrieved group against its own same-size random controls, and retain the fixed-budget experiment for comparisons at equal intervention sizes. Graph arrows indicate traversal order, not causal flow. All outputs go into a new `variable_traversal` directory.


### Variable-size selections and channel-count audit

In [ ]:
VR_DIR = ML_DIR / 'variable_traversal' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
VR_DIR.mkdir(parents=True)
VR_METHODS = ['label_constrained', 'unconstrained', 'soft_guided']
VR_NAMES = ['Label-constrained', 'Unconstrained', 'Soft-guided']
assert VR_NEIGHBORS_PER_PARENT > 0 and VR_SEED_COUNT > 0 and VR_RANDOM_REPEATS > 0 and all(c in CONCEPTS for c in VR_QUERIES)

def vr_traversal_candidates(cross, parents, scores, labels, query, threshold, method):
    if len(parents) == 0:
        return np.array([], dtype=int), np.zeros(cross.shape[0])
    retained = np.where((cross[:, parents] >= threshold) & (cross[:, parents] > 0), cross[:, parents], 0)
    strength = retained.max(axis=1)
    eligible = strength > 0
    if method == 'label_constrained':
        eligible &= np.asarray(labels) == query
    elif method not in ['unconstrained', 'soft_guided']:
        raise ValueError('Unknown traversal strategy')
    ranking = strength * scores if method == 'soft_guided' else strength
    candidates = np.flatnonzero(eligible)
    return candidates[np.lexsort((candidates, -ranking[candidates]))], ranking

check_cross = np.array([[.9], [.7], [.1]])
check_labels = ['MYO', 'LV', 'LV']
assert vr_traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'unconstrained')[0].tolist() == [0, 1]
assert vr_traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'label_constrained')[0].tolist() == [1]
assert vr_traversal_candidates(check_cross, [0], np.array([.01, .8, .9]), check_labels, 'LV', .3, 'soft_guided')[0].tolist() == [1, 0]
assert len(vr_traversal_candidates(check_cross, [], np.ones(3), check_labels, 'LV', .3, 'soft_guided')[0]) == 0
vr_sets, vr_node_rows, vr_edge_rows, vr_budget_rows = {}, [], [], []
for query in VR_QUERIES:
    c = CONCEPTS.index(query)
    final_labels = ml_labels[ML_LAYERS[-1]]
    seeds = final_labels[final_labels.label == query].sort_values(['top_iou', 'neuron'], ascending=[False, True]).head(VR_SEED_COUNT).neuron.to_numpy(dtype=int)
    if not len(seeds):
        raise ValueError(f'No final-stage seeds for {query}')
    for method in VR_METHODS:
        vr_sets[(query, method)] = {ML_LAYERS[-1]: seeds.copy()}
    # Reuse the fixed experiment's exact final-stage seeds.
    for method in VR_METHODS:
        vr_sets[(query, method)][ML_LAYERS[-1]] = tr_sets[(query, method)][ML_LAYERS[-1]].copy()
    for lower, upper in reversed(list(zip(ML_LAYERS[:-1], ML_LAYERS[1:]))):
        cross_matrix = ml_cross[('train', lower, upper)]
        for method in VR_METHODS:
            chosen, eligible_union = set(), set()
            parents = vr_sets[(query, method)][upper]
            for parent in parents:
                candidates, _ = vr_traversal_candidates(cross_matrix, [parent], ml_scores[('train', lower)][:, c], ml_labels[lower].label.to_numpy(), query, TAU, method)
                eligible_union.update(candidates.tolist())
                chosen.update(candidates[:VR_NEIGHBORS_PER_PARENT].tolist())
            vr_sets[(query, method)][lower] = np.asarray(sorted(chosen), dtype=int)
            assert len(chosen) <= len(parents) * VR_NEIGHBORS_PER_PARENT
            vr_budget_rows.append(dict(query=query, method=method, layer=lower, parents=len(parents), eligible=len(eligible_union), selected=len(chosen), total_channels=len(ml_labels[lower]), selected_fraction=len(chosen)/len(ml_labels[lower]), stopped=not bool(chosen)))
    for method in VR_METHODS:
        for layer in ML_LAYERS:
            selected = vr_sets[(query, method)][layer]
            for i in selected:
                label = ml_labels[layer].iloc[i]
                vr_node_rows.append(dict(query=query, method=method, layer=layer, neuron=int(i), node=f'{layer}:n{i}', label=label.label, train_query_iou=float(ml_scores[('train', layer)][i, c]), tie=bool(label.tie), all_zero=bool(label.all_zero)))
        for lower, upper in zip(ML_LAYERS[:-1], ML_LAYERS[1:]):
            for i in vr_sets[(query, method)][lower]:
                for j in vr_sets[(query, method)][upper]:
                    r = float(ml_cross[('train', lower, upper)][i,j])
                    if r >= TAU and r > 0:
                        vr_edge_rows.append(dict(query=query, method=method, source=f'{lower}:n{i}', target=f'{upper}:n{j}', train_r=r, eval_r=float(ml_cross[(ML_EVAL_SPLIT, lower, upper)][i,j])))
    for method in VR_METHODS:
        np.testing.assert_array_equal(vr_sets[(query, method)][ML_LAYERS[-1]], tr_sets[(query, method)][ML_LAYERS[-1]])
vr_nodes = pd.DataFrame(vr_node_rows)
vr_edges = pd.DataFrame(vr_edge_rows, columns=['query', 'method', 'source', 'target', 'train_r', 'eval_r'])
vr_budgets = pd.DataFrame(vr_budget_rows)
vr_nodes.to_csv(VR_DIR / 'selected_nodes.csv', index=False)
vr_edges.to_csv(VR_DIR / 'candidate_edges.csv', index=False)
vr_budgets.to_csv(VR_DIR / 'budget_audit.csv', index=False)
(VR_DIR / 'protocol.json').write_text(json.dumps(dict(queries=VR_QUERIES, layers=ML_LAYERS, seed_count=VR_SEED_COUNT, neighbors_per_parent=VR_NEIGHBORS_PER_PARENT, seed=SEED, threshold=TAU, random_repeats=VR_RANDOM_REPEATS, patient_index=VR_PATIENT_INDEX, split=ML_EVAL_SPLIT, checkpoint_sha256=checkpoint_sha256, budget_rule='No stage cap: union of top-b eligible neighbors per parent, separately for each method. Empty branches stop independently.', ranking='Per-parent retained correlation; soft multiplies by training query IoU.', source_run=str(ML_DIR)), indent=2))
display(vr_budgets)


### Variable-size flow graphs (your plotting style)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch

OURS = "SENCoG"      # display name only
STAGE_TAG = "Block"  # column label prefix
VR_PALETTE = {'RV': '#2d6cab', 'MYO': '#3bab3b', 'LV': '#dd6d6d'}
VR_BACKGROUND_MAX = 80
VR_RC = {'font.family': 'serif', 'font.serif': ['Times New Roman', 'Nimbus Roman', 'STIXGeneral', 'DejaVu Serif'],
         'mathtext.fontset': 'stix', 'font.size': 7, 'pdf.fonttype': 42, 'ps.fonttype': 42}
INK, GREY, CAPSULE, POOL = '#2B3440', '#9AA5B1', '#F3F5F8', '#BFC6CE'
DASH = (0, (4, 3))
vr_graph_labels = ml_nodes.set_index('node').label.to_dict()
stage_of = {l: i for i, l in enumerate(ML_LAYERS)}
N_ST = len(ML_LAYERS)
X = lambda stage: N_ST - 1 - stage   # last block is drawn leftmost

def vr__curve(p, q, bulge=None):
    (x0, y0), (x1, y1) = p, q
    if bulge is None:
        xm = (x0 + x1) / 2
        pts = [(x0, y0), (xm, y0), (xm, y1), (x1, y1)]
    else:
        pts = [(x0, y0), (x0 + bulge, y0), (x1 + bulge, y1), (x1, y1)]
    return MplPath(pts, [MplPath.MOVETO] + [MplPath.CURVE4] * 3)

def vr_draw_flow(ax, query, method, positions, context, display_edges):
    selected = {f'{l}:n{i}' for l in ML_LAYERS for i in vr_sets[(query, method)][l]}
    seeds = {f'{ML_LAYERS[-1]}:n{i}' for i in vr_sets[(query, method)][ML_LAYERS[-1]]}
    xy = lambda n: (X(stage_of[n.split(':n')[0]]), positions[n][0])
    for s, layer in enumerate(ML_LAYERS):
        x = X(s)
        ax.plot([x, x], [-1.0, 1.0], lw=15, color=CAPSULE, solid_capstyle='round', zorder=0)   # column capsule
        ax.text(x, -1.16, f'{STAGE_TAG} {s+1}', ha='center', va='top', fontsize=6.5, color=INK)
        if context[layer]:                                                                      # unselected channel pool
            pts = np.array([positions[n] for n in context[layer]])
            ax.scatter(x + .15 * pts[:, 1] / .72, pts[:, 0], s=2, color=POOL, linewidths=0, zorder=1)
    items = []
    for a, b in display_edges:
        if stage_of[a.split(':n')[0]] > stage_of[b.split(':n')[0]]:   # b = endpoint in the later block
            a, b = b, a
        e = ml_graph[a][b]
        retained = bool(np.isfinite(e['eval_r']) and e['eval_r'] >= TAU)
        same = a.split(':n')[0] == b.split(':n')[0]
        items.append((retained, same, a, b, xy(a), xy(b), np.clip(e['weight'], 0, 1)))
    for retained, same, a, b, p, q, w in sorted(items, key=lambda t: (t[0], not t[1])):
        if same:
            path = vr__curve(p, q, bulge=min(.2 + .6 * abs(q[1] - p[1]), .55))
            color, lw, alpha = GREY, .35 + .4 * w, .7
        else:
            path = vr__curve(p, q)
            color, lw, alpha = VR_PALETTE[vr_graph_labels[b]], .35 + .9 * w, .6
        ax.add_patch(PathPatch(path, fc='none', ec=color, lw=lw, ls='-' if retained else DASH,
                               alpha=alpha if retained else alpha * .55, capstyle='round', zorder=2))
    for seed in [False, True]:
        nodes = sorted(n for n in selected if (n in seeds) == seed)
        if not nodes:
            continue
        pts = np.array([xy(n) for n in nodes])
        ax.scatter(pts[:, 0], pts[:, 1], c=[VR_PALETTE[vr_graph_labels[n]] for n in nodes], s=34 if seed else 17,
                   edgecolors=INK if seed else 'white', linewidths=.7 if seed else .4, zorder=6 if seed else 5)
    ax.annotate('', xy=(N_ST - 1, 1.2), xytext=(0, 1.2), zorder=1,
                arrowprops=dict(arrowstyle='-|>', lw=.6, color=GREY, mutation_scale=6, shrinkA=0, shrinkB=0))
    ax.text((N_ST - 1) / 2, 1.25, 'traversal from seeds', ha='center', va='bottom', fontsize=6, color=GREY, style='italic')
    ax.set(xlim=(-.45, N_ST - 1 + .6), ylim=(-1.3, 1.45))
    ax.set_axis_off()

vr_display_rows, vr_layout_rows = [], []
with plt.rc_context(VR_RC):
    for query in VR_QUERIES:
        selected_union = {f'{layer}:n{i}' for method in VR_METHODS for layer in ML_LAYERS for i in vr_sets[(query,method)][layer]}
        edge_sets = {}
        for method in VR_METHODS:
            part = vr_edges[(vr_edges['query']==query)&(vr_edges.method==method)]
            edges = {tuple(sorted((r.source,r.target))) for r in part.itertuples()}
            # Same display rule as the original: one strongest within-stage neighbour per selected channel.
            for layer in ML_LAYERS:
                channels = vr_sets[(query,method)][layer]
                for i in channels:
                    node = f'{layer}:n{i}'
                    neighbors = [f'{layer}:n{j}' for j in channels if i!=j and ml_graph.has_edge(node,f'{layer}:n{j}')]
                    if neighbors:
                        neighbor = min(neighbors,key=lambda n:(-ml_graph[node][n]['weight'],n))
                        edges.add(tuple(sorted((node,neighbor))))
            selected = {f'{layer}:n{i}' for layer in ML_LAYERS for i in vr_sets[(query,method)][layer]}
            assert all(a in selected and b in selected and ml_graph.has_edge(a,b) for a,b in edges)
            edge_sets[method] = edges
            for a,b in sorted(edges):vr_display_rows.append(dict(query=query,method=method,source=a,target=b,**ml_graph[a][b]))
        union_edges = set().union(*edge_sets.values())
        positions,context,orders = {},{},{}
        for stage,layer in enumerate(ML_LAYERS):
            ids = sorted(ml_nodes.loc[ml_nodes.layer==layer,'node'])
            for k,node in enumerate(ids):
                radius = np.sqrt((k+.5)/max(len(ids),1));angle = k*np.pi*(3-np.sqrt(5))
                positions[node] = (.98*radius*np.cos(angle),.72*radius*np.sin(angle),1.45*stage)
            others = [n for n in ids if n not in selected_union]
            if len(others)>VR_BACKGROUND_MAX:
                keep = np.sort(np.random.default_rng(SEED+7919*(stage+1)).choice(len(others),VR_BACKGROUND_MAX,replace=False))
                others = [others[i] for i in keep]
            context[layer] = others
        orders[ML_LAYERS[-1]] = [f'{ML_LAYERS[-1]}:n{i}' for i in vr_sets[(query,VR_METHODS[0])][ML_LAYERS[-1]]]
        for lower,upper in reversed(list(zip(ML_LAYERS[:-1],ML_LAYERS[1:]))):
            def barycentre(node):
                ranks = [j for j,n in enumerate(orders[upper]) if tuple(sorted((node,n))) in union_edges]
                return np.mean(ranks) if ranks else float('inf')
            orders[lower] = sorted([n for n in selected_union if n.split(':n')[0]==lower],key=lambda n:(barycentre(n),n))
        for stage,layer in enumerate(ML_LAYERS):
            ordered = orders[layer]
            for j,(node,x) in enumerate(zip(ordered,np.linspace(-.82,.82,len(ordered)) if len(ordered)>1 else [0.])):
                positions[node] = (float(x),float(.21*np.sin(j*2.39996+stage)),1.45*stage)
        for node,(x,y,z) in positions.items():vr_layout_rows.append(dict(query=query,node=node,x=x,y=y,z=z,context=any(node in v for v in context.values())))

        handles = [Line2D([], [], marker='o', ls='None', markersize=4, markerfacecolor=c, markeredgecolor='white',
                          markeredgewidth=.4, label=k) for k, c in VR_PALETTE.items()]
        handles += [Line2D([], [], marker='o', ls='None', markersize=5.5, markerfacecolor=VR_PALETTE[query],
                           markeredgecolor=INK, markeredgewidth=.7, label=f'{query} seeds'),
                    # Line2D([], [], color=INK, lw=.9, label=r'$r\geq\tau$'),
                    # Line2D([], [], color=INK, lw=.9, ls=DASH, label=r'$r<\tau$')
                    ]
        legend_kw = dict(handles=handles, loc='upper center', frameon=False, fontsize=7, handletextpad=.2, columnspacing=1.1)

        fig = plt.figure(figsize=(6.9, 2.8))
        gs = fig.add_gridspec(1, 3, left=.005, right=.995, top=.88, bottom=.1, wspace=.03)
        for col, (method, name) in enumerate(zip(VR_METHODS, VR_NAMES)):
            ax = fig.add_subplot(gs[0, col])
            vr_draw_flow(ax, query, method, positions, context, edge_sets[method])
            ax.text(.5, -.01, f'({chr(97+col)}) {name}', transform=ax.transAxes, ha='center', va='top', fontsize=8)
        fig.legend(bbox_to_anchor=(.5, 1.01), ncol=6, **legend_kw)
        for ext in ['png', 'pdf', 'svg']:
            fig.savefig(VR_DIR/f'{query}_traversal_graphs.{ext}', dpi=300, bbox_inches='tight', pad_inches=.03, facecolor='white')
        plt.show()

        for method, name in zip(VR_METHODS, VR_NAMES):
            fig = plt.figure(figsize=(3.4, 3.0))
            ax = fig.add_axes([.01, .12, .98, .70])
            vr_draw_flow(ax, query, method, positions, context, edge_sets[method])
            ax.text(.5, -.01, f'{query} · {name}', transform=ax.transAxes, ha='center', va='top', fontsize=8)
            fig.legend(bbox_to_anchor=(.5, 1.01), ncol=3, **legend_kw)
            for ext in ['png', 'pdf', 'svg']:
                fig.savefig(VR_DIR/f'{query}_{method}_flow.{ext}', dpi=300, bbox_inches='tight', pad_inches=.03, facecolor='white')
            plt.close(fig)

pd.DataFrame(vr_display_rows).to_csv(VR_DIR/'graph_display_edges.csv',index=False)
pd.DataFrame(vr_layout_rows).to_csv(VR_DIR/'graph_layout.csv',index=False)
(VR_DIR/'GRAPH_CAPTION.md').write_text(f'Concept-guided cross-stage {OURS} neighbourhoods retrieved using label-constrained, unconstrained and soft-guided traversal. Blocks are ordered right to left from Block 1 to Block {N_ST}, so the final-block seeds sit leftmost and traversal proceeds rightwards to earlier blocks; columns share node coordinates; selected channel counts vary by strategy. Node colours indicate assigned concepts and outlined nodes are final-block seeds. Cross-stage edges take the colour of their endpoint in the later block; grey arcs are within-block neighbours. Solid/dashed lines indicate evaluation correlations above/below the training threshold; grey dots are unselected channels.\n\nDisplay notes: all retrieved adjacent-stage edges are shown, with one strongest within-stage neighbour per selected channel as context. Background nodes are sampled from channels outside the union of selected sets. Coordinates are schematic; edges remain statistical associations, not intervention-measured dependencies.\n')
print('Flow figures:',VR_DIR)

### Same-patient variable-size responses

In [ ]:
assert 0 <= VR_PATIENT_INDEX < len(images[ML_EVAL_SPLIT])
vr_pid = str(manifests[ML_EVAL_SPLIT].iloc[VR_PATIENT_INDEX].pid)
vr_heat_rows = []
for query in VR_QUERIES:
    fig, axes = plt.subplots(3, len(ML_LAYERS), figsize=(12, 7), squeeze=False)
    for row, (method, method_name) in enumerate(zip(VR_METHODS, VR_NAMES)):
        for col, layer in enumerate(ML_LAYERS):
            ax = axes[row, col]
            selected = vr_sets[(query, method)][layer]
            if not len(selected):
                ax.text(.5, .5, 'No eligible group', ha='center'); ax.axis('off'); continue
            train_maps = ml_acts['train'][layer]
            scales = train_maps.mean((2,3)).std(dim=0).numpy()
            valid = selected[scales[selected] > 1e-8]
            if not len(valid):
                ax.text(.5, .5, 'Constant channels', ha='center'); ax.axis('off'); continue
            # Common training-derived per-channel calibration; display only, never used for selection.
            group = (ml_acts[ML_EVAL_SPLIT][layer][VR_PATIENT_INDEX, valid] / torch.as_tensor(scales[valid])[:,None,None]).mean(0)
            reference = (train_maps[:,valid] / torch.as_tensor(scales[valid])[None,:,None,None]).mean(1).numpy()
            vmax = max(float(np.percentile(reference, 99)), 1e-8)
            heat = torch.nn.functional.interpolate(group[None,None], size=images[ML_EVAL_SPLIT].shape[-2:], mode='bilinear', align_corners=False)[0,0].numpy()
            ed, es = images[ML_EVAL_SPLIT][VR_PATIENT_INDEX].numpy()
            paired = np.concatenate([ed, es], axis=1)
            ax.imshow(paired, cmap='gray')
            ax.imshow(np.concatenate([heat,heat], axis=1), cmap='magma', vmin=0, vmax=vmax, alpha=.45)
            ax.set_title(f'{layer.split(".")[-1]} · {len(selected)} channels', fontsize=9)
            ax.set_xlabel('ED | ES', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            if col == 0:ax.set_ylabel(method_name, fontsize=9)
            vr_heat_rows.append(dict(query=query, method=method, layer=layer, pid=vr_pid, channels=selected.tolist(), displayed_channels=valid.tolist(), display_vmax=vmax))
    fig.suptitle(f'{query} · patient {vr_pid} · joint ED/ES activations', fontsize=11)
    fig.tight_layout()
    for ext in ['png','pdf']:
        fig.savefig(VR_DIR / f'{query}_patient_{VR_PATIENT_INDEX}_stages.{ext}', dpi=250, bbox_inches='tight')
    plt.show()
(VR_DIR / 'patient_view_provenance.json').write_text(json.dumps(vr_heat_rows, indent=2))


### Variable-size group interventions and size-matched random controls

In [ ]:
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == checkpoint_sha256
model.eval()
vr_inputs = images[ML_EVAL_SPLIT]
vr_ids = manifests[ML_EVAL_SPLIT].pid.astype(str).tolist()
vr_effects, vr_details, vr_skipped = [], [], []
vr_rng = np.random.default_rng(SEED)
for lower, upper in zip(ML_LAYERS[:-1], ML_LAYERS[1:]):
    means = ml_acts['train'][lower].mean((0,2,3)).numpy()
    target_scale = ml_acts['train'][upper].mean((2,3)).numpy().std(axis=0, ddof=1)
    clean, target_clean = run_stage_intervention(model, vr_inputs, lower, upper, [], 'zero', BATCH_SIZE, DEVICE)
    np.testing.assert_allclose(target_clean, ml_acts[ML_EVAL_SPLIT][upper].mean((2,3)).numpy(), rtol=1e-4, atol=1e-5)
    predicted = clean.argmax(1)
    clean_logp = clean.log_softmax(1)
    idx = torch.arange(len(vr_ids))
    cache = {}
    for query in VR_QUERIES:
        random_by_method = {}
        for method in VR_METHODS:
            k = len(vr_sets[(query, method)][lower])
            if not k:
                vr_skipped.append(dict(query=query, method=method, source_layer=lower, reason='Empty variable-size source group'))
                continue
            random_by_method[method] = [np.sort(vr_rng.choice(len(means), k, replace=False)) for _ in range(VR_RANDOM_REPEATS)]
        for baseline in ['zero','train_mean']:
            for method in VR_METHODS:
                if method not in random_by_method:continue
                random_groups = random_by_method[method]
                sources = vr_sets[(query,method)][lower]
                targets = vr_sets[(query,method)][upper]
                valid_targets = targets[target_scale[targets] > 1e-8]
                groups = [('retrieved',0,sources)] + [('random',r,g) for r,g in enumerate(random_groups)]
                for selection, repeat, group in groups:
                    key = (baseline,tuple(sorted(group.tolist())))
                    if key not in cache:
                        cache[key] = run_stage_intervention(model, vr_inputs, lower, upper, group, baseline, BATCH_SIZE, DEVICE, source_means=means)
                    changed, target_after = cache[key]
                    kl = (clean_logp.exp()*(clean_logp-changed.log_softmax(1))).sum(1).clamp_min(0).numpy()
                    delta = (target_clean[:,valid_targets]-target_after[:,valid_targets])/target_scale[valid_targets][None,:]
                    target_effect = np.abs(delta).mean(1) if len(valid_targets) else np.full(len(vr_ids),np.nan)
                    c = CONCEPTS.index(query)
                    evidence = ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,c].mean(1)
                    other = np.delete(ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,:],c,axis=2).max(2)
                    selectivity = (ml_profiles[(ML_EVAL_SPLIT,lower)]['per_patient_iou'][:,group,c]-other).mean(1)
                    after_prob = changed.softmax(1)[idx,predicted].numpy()
                    for patient,pid in enumerate(vr_ids):
                        vr_effects.append(dict(query=query, method=method, source_layer=lower, target_layer=upper, baseline=baseline, selection=selection, repeat=repeat, pid=pid, source_k=len(group), target_k=len(valid_targets), kl=float(kl[patient]), flip=float(changed[patient].argmax()!=predicted[patient]), target_effect=float(target_effect[patient]), query_iou=float(evidence[patient]), selectivity=float(selectivity[patient]), probability_before=float(clean.softmax(1)[patient,predicted[patient]]), probability_after=float(after_prob[patient]), predicted_class=list(cfg.classes)[int(predicted[patient])], sources=json.dumps(group.tolist()), targets=json.dumps(valid_targets.tolist())))
                        if selection=='retrieved':
                            for pos,j in enumerate(valid_targets):
                                vr_details.append(dict(query=query,method=method,baseline=baseline,pid=pid,source_layer=lower,target_layer=upper,target=int(j),signed_change_sd=float(delta[patient,pos])))
            print(query, lower, '→', upper, baseline, flush=True)
    restored,_ = run_stage_intervention(model,vr_inputs,lower,upper,[],'zero',BATCH_SIZE,DEVICE)
    torch.testing.assert_close(restored,clean)
    pd.DataFrame(vr_effects).to_csv(VR_DIR/'patient_effects.csv',index=False)
vr_results = pd.DataFrame(vr_effects)
pd.DataFrame(vr_details).to_csv(VR_DIR/'downstream_channel_effects.csv',index=False)
(VR_DIR/'skipped.json').write_text(json.dumps(vr_skipped,indent=2))
if vr_results.empty:
    raise RuntimeError('No nonempty variable-size source groups. See budget_audit.csv.')


### Variable-size summaries (interpret alongside channel counts)

In [ ]:
def vr_patient_interval(values, repeats, seed):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (float(values.mean()) if len(values) else np.nan, np.nan, np.nan, len(values))
    draw = np.random.default_rng(seed).integers(0, len(values), size=(repeats, len(values)))
    lower, upper = np.quantile(values[draw].mean(1), [.025, .975])
    return float(values.mean()), float(lower), float(upper), len(values)

vr_metrics = ['kl','flip','target_effect','query_iou','selectivity']
vr_keys = ['query','method','source_layer','target_layer','baseline','selection','pid']
vr_patient = vr_results.groupby(vr_keys,as_index=False)[vr_metrics].mean()
vr_summary_rows = []
for keys,group in vr_patient.groupby(vr_keys[:-1]):
    for metric in vr_metrics:
        mean,lo,hi,n = vr_patient_interval(group[metric],BOOTSTRAPS,SEED)
        vr_summary_rows.append(dict(zip(vr_keys[:-1],keys),metric=metric,mean=mean,ci_low=lo,ci_high=hi,patients=n))
vr_summary = pd.DataFrame(vr_summary_rows)
vr_summary.to_csv(VR_DIR/'summary.csv',index=False)
vr_paired_rows = []
for keys,group in vr_patient.groupby(['query','source_layer','baseline']):
    for metric in vr_metrics:
        table = group.pivot(index='pid',columns=['method','selection'],values=metric).reindex(columns=pd.MultiIndex.from_product([VR_METHODS,['retrieved','random']]))
        comparisons = [(f'{m} minus random',table[(m,'retrieved')]-table[(m,'random')]) for m in VR_METHODS]
        comparisons += [(f'{a} minus {b}',table[(a,'retrieved')]-table[(b,'retrieved')]) for a,b in [('soft_guided','unconstrained'),('label_constrained','unconstrained'),('soft_guided','label_constrained')]]
        for comparison,values in comparisons:
            mean,lo,hi,n = vr_patient_interval(values,BOOTSTRAPS,SEED)
            vr_paired_rows.append(dict(query=keys[0],source_layer=keys[1],baseline=keys[2],comparison=comparison,metric=metric,mean=mean,ci_low=lo,ci_high=hi,patients=n))
vr_paired = pd.DataFrame(vr_paired_rows)
vr_paired.to_csv(VR_DIR/'paired_comparisons.csv',index=False)
for query in VR_QUERIES:
    for layer in vr_summary.source_layer.unique():
        fig,axes = plt.subplots(1,2,figsize=(6.2,2.8))
        for ax,metric,title in zip(axes,['kl','flip'],['KL divergence','Prediction flips']):
            for baseline,offset,marker in [('zero',-.1,'o'),('train_mean',.1,'s')]:
                part = vr_summary[(vr_summary['query']==query)&(vr_summary.source_layer==layer)&(vr_summary.baseline==baseline)&(vr_summary.selection=='retrieved')&(vr_summary.metric==metric)].set_index('method').reindex(VR_METHODS)
                x = np.arange(3)+offset
                ax.vlines(x,part.ci_low,part.ci_high,color='#218A78' if baseline=='zero' else '#6486B5')
                ax.plot(x,part['mean'],marker,linestyle='None',label='Zeroing' if baseline=='zero' else 'Training mean')
            ax.set(xticks=range(3),xticklabels=['Constrained','Unconstrained','Soft-guided'],title=title)
            ax.tick_params(axis='x',labelsize=8,rotation=15)
            ax.spines[['top','right']].set_visible(False)
            if metric=='flip':ax.set_ylim(-.025,1.025);ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f'{v:.0%}'))
        axes[0].legend(frameon=False,fontsize=7)
        fig.suptitle(f'{query} · intervention at {layer.split(".")[-1]}',fontsize=10)
        fig.tight_layout()
        for ext in ['png','pdf','svg']:fig.savefig(VR_DIR/f'{query}_{layer}_comparison.{ext}',dpi=300,bbox_inches='tight')
        plt.show()
display(vr_paired)
vr_local = vr_results[(vr_results.pid==vr_pid)&(vr_results.selection=='retrieved')]
statements = []
for r in vr_local.itertuples():
    statements.append(f'{r.query}, {r.method}, {r.source_layer} → {r.target_layer}, {r.baseline}: intervening on {r.source_k} channels changed the original {r.predicted_class} probability from {r.probability_before:.3f} to {r.probability_after:.3f}; KL={r.kl:.5f}, flip={bool(r.flip)}, mean absolute downstream change={r.target_effect:.3f} training SD.')
notes = 'Variable source and target budgets differ between methods; raw KL/flip contrasts are descriptive and do not establish superiority at equal intervention size. Each method has its own size-matched random source baseline. Selection is global and training-derived; the displayed input is fixed across stages. Evaluation masks assess overlap only. Labels are forced argmax assignments, not verified detectors. Heatmap colour scales are training-calibrated per group and not directly comparable across groups. Downstream targets differ between strategies, so target-effect contrasts describe different retrieved groups, not a common target. Random source controls use the same downstream targets for each strategy. No intervention validates an individual correlation edge. Intervals are exploratory, pointwise patient-bootstrap intervals conditional on this model and selection; no multiple-testing correction. Zero and mean replacement can create atypical internal states.'
(VR_DIR/'RESULTS.md').write_text('# Variable-size traversal comparison\n\n'+notes+'\n\n## Prespecified patient '+vr_pid+'\n\n'+'\n\n'.join(statements)+'\n\n## Paired comparisons\n\n```\n'+vr_paired.to_string(index=False)+'\n```\n')
print('Traversal results:',VR_DIR)

vr_sizes = vr_results.groupby(['query','method','source_layer'],as_index=False).agg(source_k=('source_k','first'),target_k=('target_k','first'))
vr_sizes.to_csv(VR_DIR/'intervention_sizes.csv',index=False)
vr_summary = vr_summary.merge(vr_sizes,on=['query','method','source_layer'],validate='many_to_one')
vr_summary.to_csv(VR_DIR/'summary.csv',index=False)
display(vr_sizes)
